# Partie B — Le marché

Source des données : Inside Airbnb (licence CC BY 4.0), snapshot du 17/09/2026 (date de collecte `last_scraped` : 19/09/2026, vérifiée en A).
Ville : Lyon.
Responsable : Membre 2 (B1, B2, B3 Q1, Q2, Q4). Assemblage des notebooks en un seul fichier : Membre 3.

> Ce notebook lit les **tables nettoyées du Membre 1** (`data_clean/listings_propre.parquet` et `reviews_propre.parquet`), jamais les fichiers bruts. Les étapes s'enchaînent en mémoire : B1, B2, puis Q1, Q2, Q4. Les exports intermédiaires (CSV, HTML) sont écrits dans `data_clean/sorties_b/`, dossier non versionné.
> **Ajouts du Membre 3, à relire par le Membre 2 :** la source externe de population (fin de B2, données Insee) et B3 Q3 (où investir).
> **À compléter :** B4 (6 graphiques Matplotlib ou Seaborn).

| Tâche | Contenu |
|---|---|
| B1 | Création de variables |
| B2 | Agrégation des avis et jointure contrôlée |
| B3 Q1 | Annonces actives et saisonnalité |
| B3 Q2 | Hôtes multi-annonces, concentration, Lorenz et Gini |
| B3 Q3 | Où investir |
| B3 Q4 | Licences et cadre légal |
| B4 | Visualisations exploratoires (à compléter) |

In [ ]:
import sys
sys.path.append("..")
from src.config import *
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
np.random.seed(SEED)

# Référence commune à toutes les sections
DATE_REFERENCE = pd.Timestamp("2026-09-19")   # last_scraped : date de collecte (le snapshot annoncé est le 17/09/2026)
SOURCE_NETTOYEE = True                          # les tables lues sont celles du Membre 1
STATUT = "validee"
SORTIES_B = CLEAN / "sorties_b"
SORTIES_B.mkdir(exist_ok=True)

listings_propre = pd.read_parquet(CLEAN / "listings_propre.parquet")
reviews_propre = pd.read_parquet(CLEAN / "reviews_propre.parquet", columns=["id", "listing_id", "date"])
print("listings :", listings_propre.shape, "| avis :", reviews_propre.shape)

## B1 — Création de variables

Créer des variables utiles pour Lyon
**Projet : location de courte durée · Membre 2**

Ce notebook crée **neuf variables** pour les analyses d'activité, de concentration
des annonces et de prix/revenus. 



Source : Inside Airbnb, CC BY 4.0 ; données altérées pour le cours(petit rappel).
Snapshot annoncé : **17 septembre 2026**. Le champ `last_scraped` du fichier fourni
indique **19 septembre 2026** : cette différence est signalée, sans réécrire les dates.
Les avis des douze derniers mois sont ceux de la colonne fournie au moment de la collecte.
Une fenêtre arrêtée strictement au 17 septembre devra être recalculée depuis `reviews` en B2.


In [ ]:
# Données : tables nettoyées du Membre 1 (A6). Elles sont converties en texte pour conserver
# les contrôles de format de B1 ; les formats exotiques de prix ont déjà été traités en A3 et A5.
DOSSIER = SORTIES_B

# Hypothèses explicites, modifiables pour une analyse de sensibilité.
TAUX_AVIS = 0.50
DUREE_SEJOUR_DEFAUT = 3
PLAFOND_OCCUPATION = 0.70
JOURS_ANNEE = 365
CAPACITE_MAX_PROVISOIRE = 16
PRIX_MAX_PLAUSIBLE = 9999

source = listings_propre.drop(columns=["amenities"]).copy()   # liste : non hachable
source["last_scraped"] = DATE_REFERENCE.strftime("%Y-%m-%d")
for colonne in ["id", "host_id", "price", "accommodates", "minimum_nights", "number_of_reviews_ltm",
                "availability_365", "estimated_occupancy_l365d", "estimated_revenue_l365d"]:
    source[colonne] = source[colonne].astype("string")

colonnes_requises = [
    "id", "host_id", "last_scraped", "price", "accommodates",
    "minimum_nights", "number_of_reviews_ltm", "availability_365"
]
absentes = sorted(set(colonnes_requises) - set(source.columns))
if absentes:
    raise ValueError(f"Colonnes nécessaires absentes : {absentes}")

df = source[colonnes_requises].copy()
# Colonnes d'origine : ne pas utiliser les estimations recalculées en B1.
for colonne in ["estimated_occupancy_l365d", "estimated_revenue_l365d"]:
    df[colonne] = source[colonne]

df.insert(0, "ligne_source", range(1, len(df) + 1))
print("Lignes :", len(source), "| Annonces distinctes :", source["id"].nunique())
print("Doublons exacts sur toutes les colonnes :", source.duplicated().sum())
print("Lignes en plus de la première par id :", source["id"].duplicated().sum())
print("Dates de collecte :", sorted(source["last_scraped"].dropna().unique().tolist()))

if SOURCE_NETTOYEE and (source["id"].isna().any() or source["id"].duplicated().any()):
    raise ValueError("La table annoncée propre contient des id manquants ou répétés : revoir A5.")

### 2. Préparer les entrées et récupérer les prix vérifiables
Le dictionnaire définit `price` en monnaie locale (euros à Lyon) ; le symbole `$` standard
est un artefact d'export. On traite les formats numériques anglais et français.

**Récupération ciblée :** les libellés `CAD` / `dollars` et les prix négatifs sont comparés
au ratio `estimated_revenue_l365d / estimated_occupancy_l365d`, deux colonnes **d'origine**
portant sur 365 jours. On suppose que le revenu source est en euros : cette hypothèse
est étayée par la concordance avec les prix standard, sans constituer une preuve externe.
Il ne s'agit pas d'une conversion CAD/EUR avec un taux de change.

La récupération exige des nuits et un revenu positifs et finis, au maximum 365 nuits,
un prix reconstitué positif inférieur à 9 999 € et un montant compatible avec le texte initial.
La tolérance est de 1 € sur le revenu source, plus 0,005 € par nuit pour un montant à
centimes ou 0,50 € par nuit pour un libellé monétaire entier (arrondi possible à l'euro).
Le prix récupéré est le ratio arrondi au centime : **reconstruction approximative**, à valider
avec le membre 1 en A5. Un contrôle affiche aussi les éventuels désaccords sur les prix standard.

Les candidats non vérifiables restent manquants, notamment si les nuits valent zéro.
Les prix absents et nuls ne sont pas imputés. Les prix >= 9 999 € sont masqués selon
le seuil provisoire retenu. `price` garde toujours sa valeur originale ; les colonnes
`prix_corrige`, `prix_justification` et `prix_eur_avant_recuperation` documentent le traitement.

Les capacités hors 1–16 sont masquées uniquement pour le prix par personne (seuil provisoire).
Les minimums de nuits hors 1–365, non entiers ou absents ne servent pas au modèle annuel.
Les minimums supérieurs à 30 nuits sont signalés car le modèle touristique est moins adapté.
Aucune annonce n'est supprimée ; le traitement des autres extrêmes reste à effectuer en A5.


In [ ]:
prix_texte = df['price'].str.strip()
prix_sans_symbole = (prix_texte.str.replace('$', '', regex=False)
                    .str.replace('€', '', regex=False).str.strip())

# Exemple : $1,234.50 -> 1234.50 ; 61,00 $ -> 61.00.
format_anglais = prix_sans_symbole.str.fullmatch(
    r'-?(?:[0-9]{1,3}(?:,[0-9]{3})+|[0-9]+)(?:\.[0-9]{1,2})?', na=False
)
format_francais = prix_sans_symbole.str.fullmatch(r'-?[0-9]+,[0-9]{1,2}', na=False)
prix_normalise = prix_sans_symbole.where(format_anglais | format_francais)
prix_normalise.loc[format_anglais] = prix_sans_symbole.loc[format_anglais].str.replace(',', '', regex=False)
prix_normalise.loc[format_francais] = prix_sans_symbole.loc[format_francais].str.replace(',', '.', regex=False)
prix_nombre = pd.to_numeric(prix_normalise, errors='coerce').astype('Float64')
df['prix_eur'] = prix_nombre.where(prix_nombre.gt(0))

df['prix_statut'] = 'utilisable'
df.loc[prix_texte.isna(), 'prix_statut'] = 'manquant'
df.loc[prix_texte.notna() & prix_nombre.isna(), 'prix_statut'] = 'format_ou_devise_a_verifier'
df.loc[prix_nombre.le(0).fillna(False), 'prix_statut'] = 'non_positif'

capacite = pd.to_numeric(df['accommodates'], errors='coerce').astype('Float64')
minimum = pd.to_numeric(df['minimum_nights'], errors='coerce').astype('Float64')
avis = pd.to_numeric(df['number_of_reviews_ltm'], errors='coerce').astype('Float64')
disponibilite = pd.to_numeric(df['availability_365'], errors='coerce').astype('Float64')

df['capacite_utilisable'] = capacite.where(
    capacite.between(1, CAPACITE_MAX_PROVISOIRE) & capacite.mod(1).eq(0)
)
df['minimum_nuits_utilisable'] = minimum.where(
    minimum.between(1, JOURS_ANNEE) & minimum.mod(1).eq(0)
)
df['avis_12m_utilisables'] = avis.where(avis.ge(0) & avis.mod(1).eq(0))
df['disponibilite_365_utilisable'] = disponibilite.where(
    disponibilite.between(0, JOURS_ANNEE) & disponibilite.mod(1).eq(0)
)
df['minimum_superieur_30'] = minimum.gt(30)
df['id_repete'] = df['id'].duplicated(keep=False)

# Récupération ciblée des libellés CAD / dollars et des prix négatifs.
# Hypothèse de travail : le revenu source est en euros et correspond aux nuits source.
# Elle est étayée par les prix standard du fichier, pas par une preuve externe.
# Les deux champs source portent sur l365d : même fenêtre de 365 jours.
nuits_source = pd.to_numeric(df['estimated_occupancy_l365d'], errors='coerce').astype('Float64')
revenu_source = pd.to_numeric(df['estimated_revenue_l365d'], errors='coerce').astype('Float64')
source_valide = (nuits_source.gt(0) & nuits_source.le(365)
                 & revenu_source.gt(0) & revenu_source.lt(float('inf'))).fillna(False)
df['prix_reconstitue_source'] = (revenu_source / nuits_source.where(source_valide)).round(2)

# Contrôle de cohérence sur les prix standard avant toute correction.
reference = (df['prix_statut'].eq('utilisable') & df['prix_eur'].lt(PRIX_MAX_PLAUSIBLE)
             & source_valide).fillna(False)
erreur_reference = (revenu_source - df['prix_eur'] * nuits_source).abs()
# Tolérance : 1 euro sur le revenu source et un demi-centime sur le tarif affiché.
accord_reference = erreur_reference.le(1 + 0.005 * nuits_source).fillna(False)
print('Cohérence revenu / nuits sur les prix standard :',
      int((reference & accord_reference).sum()), '/', int(reference.sum()))

# Ne reconnaître que des formats précis ; ne pas extraire arbitrairement des chiffres.
montant_cad = prix_texte.str.extract(r'(?i)^CAD\s+([0-9]+(?:[.,][0-9]{1,2})?)$', expand=False)
montant_dollars = prix_texte.str.extract(r'(?i)^([0-9]+(?:[.,][0-9]{1,2})?)\s+dollars?$', expand=False)
libelle_devise = montant_cad.notna() | montant_dollars.notna()
montant_texte = montant_cad.fillna(montant_dollars).str.replace(',', '.', regex=False)
montant_libelle = pd.to_numeric(montant_texte, errors='coerce').astype('Float64')
negatif = prix_nombre.lt(0).fillna(False)
candidat = libelle_devise | negatif
montant_compare = montant_libelle.where(libelle_devise, prix_nombre.abs())

# Un libellé entier peut avoir été arrondi à l'euro : tolérance de 0,50 euro/nuit.
# Les montants à décimales sont comparés avec une tolérance de 0,005 euro/nuit.
precision_prix = pd.Series(0.005, index=df.index, dtype='Float64')
libelle_entier = libelle_devise & ~montant_texte.str.contains(r'[.,]', regex=True, na=False)
precision_prix.loc[libelle_entier] = 0.5
accord_montant = (revenu_source - montant_compare * nuits_source).abs().le(
    1 + precision_prix * nuits_source
).fillna(False)

recuperable = (candidat & source_valide & accord_montant
               & df['prix_reconstitue_source'].gt(0)
               & df['prix_reconstitue_source'].lt(PRIX_MAX_PLAUSIBLE)).fillna(False)
df['prix_corrige'] = recuperable
# Garder le prix avant récupération pour retracer les changements.
df['prix_eur_avant_recuperation'] = df['prix_eur'].copy()
df['prix_justification'] = 'Aucune récupération appliquée'
df.loc[candidat, 'prix_justification'] = 'Récupération non validée : incohérence ou prix hors seuil'
df.loc[candidat & ~source_valide, 'prix_justification'] = 'Revenu/nuits source absents, invalides ou nuls'
df.loc[recuperable, 'prix_eur'] = df.loc[recuperable, 'prix_reconstitue_source']
df.loc[recuperable, 'prix_statut'] = 'recupere_coherence_interne'
df.loc[recuperable, 'prix_justification'] = (
    'Revenu source / nuits source, arrondi au centime ; accord du montant selon précision ; '
    'hypothèse de revenu source en EUR, à valider en A5'
)

# Masquer également les prix extrêmes ; conserver la colonne price d'origine.
prix_extreme = df['prix_eur'].ge(PRIX_MAX_PLAUSIBLE).fillna(False)
df.loc[prix_extreme, 'prix_statut'] = 'prix_extreme_a_verifier'
df.loc[prix_extreme, 'prix_justification'] = 'Seuil provisoire : prix >= 9 999 EUR'
df.loc[prix_extreme, 'prix_eur'] = pd.NA

assert df.loc[candidat & ~recuperable, 'prix_eur'].isna().all()
assert not df['prix_eur'].ge(PRIX_MAX_PLAUSIBLE).fillna(False).any()
assert df['price'].equals(source['price']), 'Le prix original a été modifié.'
print('Prix candidats à la récupération :', int(candidat.sum()))
print('Prix récupérés (lignes, doublons compris) :', int(recuperable.sum()))
print('Prix candidats restant masqués :', int((candidat & ~recuperable).sum()))
print('Prix extrêmes masqués :', int(prix_extreme.sum()))

print('Statut des prix (lignes, doublons compris) :')
print(df['prix_statut'].value_counts().to_string())
print('Capacités non utilisables :', df['capacite_utilisable'].isna().sum())
print('Minimum de nuits non utilisable :', df['minimum_nuits_utilisable'].isna().sum())
print('Minimum supérieur à 30 nuits :', df['minimum_superieur_30'].sum())


### 3. Cinq variables descriptives
| Variable | Construction | Utilité / interprétation |
|---|---|---|
| `prix_par_personne` | prix / capacité utilisable | Comparer le prix affiché à pleine capacité ; ce n'est pas une facture réelle par voyageur. |
| `nb_annonces_hote_local` | nombre d'id distincts par `host_id` dans ce fichier | Étudier la concentration des annonces, sans compter plusieurs fois un id répété. Ce n'est pas le portefeuille mondial de l'hôte. |
| `hote_multi_annonce` | 1 si nombre d'annonces ≥ 2, sinon 0 | Comparer hôtes mono et multi-annonces ; plusieurs annonces ne prouvent pas un statut professionnel. |
| `activite_avis_12m` | 1 si au moins un avis sur douze mois, sinon 0 | Indicateur indirect d'activité récente. Zéro signifie absence d'avis, pas preuve d'inactivité. |
| `taux_disponibilite_365` | jours disponibles / 365 | Part disponible sur les 365 jours futurs ; la valeur est entre 0 et 1. |

**La disponibilité future n'est pas l'occupation passée.** On ne calcule pas les nuits
louées avec `365 - availability_365` : l'indisponibilité inclut des blocages de l'hôte.


In [ ]:
# Vérifier l'attribution d'un id à un seul hôte avant de compter.
id_valide = df['id'].str.fullmatch(r'[1-9][0-9]*', na=False)
hote_valide = df['host_id'].str.fullmatch(r'[1-9][0-9]*', na=False)
conflits_hotes = (df.loc[id_valide].groupby('id')['host_id']
                 .nunique(dropna=False).gt(1))
if conflits_hotes.any():
    raise ValueError('Un même id est associé à plusieurs hôtes : résoudre ces conflits en A5.')

comptage_hotes = (df.loc[id_valide & hote_valide]
                 .groupby('host_id')['id'].nunique())

df['prix_par_personne'] = df['prix_eur'] / df['capacite_utilisable']
df['nb_annonces_hote_local'] = df['host_id'].map(comptage_hotes).astype('Int64')
df['hote_multi_annonce'] = df['nb_annonces_hote_local'].ge(2).astype('Int64')
df['activite_avis_12m'] = df['avis_12m_utilisables'].gt(0).astype('Int64')
df['taux_disponibilite_365'] = df['disponibilite_365_utilisable'] / JOURS_ANNEE


### 4. Quatre variables estimées et leurs hypothèses
Nous construisons un modèle explicite inspiré de la méthode publiée par Inside Airbnb :
**50 %** des séjours donnent un avis ; durée de référence **3 nuits**, relevée au minimum
de l'annonce si celui-ci est supérieur ; occupation limitée à **70 %** de 365 jours.
Ces paramètres ne sont pas des mesures démontrées pour Lyon : ils sont à discuter et à faire varier.

Avec `A` le nombre d'avis sur douze mois, `m` le minimum de nuits utilisable et `p` le prix :

| Variable | Formule |
|---|---|
| `sejours_estimes_avant_plafond` | A / 0,50 |
| `nuits_estimees_12m` | min[(A / 0,50) × max(3, m), 365 × 0,70] |
| `taux_occupation_estime_12m` | nuits estimées / 365 |
| `revenu_brut_estime_12m` | nuits estimées × prix affiché |

Le nombre de séjours reste une estimation **avant** plafonnement : une fois le plafond
atteint, les nuits ne lui sont plus proportionnelles. Un nombre fractionnaire de nuits
est une espérance statistique, pas un calendrier reconstitué. Le plafond est **255,5 nuits**
dans notre formule ; il n'est pas arrondi avant le calcul du revenu.

Le revenu est un scénario brut sur douze mois au prix observé. Il ignore variations
de prix, remises, frais, charges et fiscalité. Il n'est ni le revenu réellement encaissé,
ni le bénéfice, ni une prévision fiable à lui seul. Une annonce nouvellement créée n'a
pas nécessairement été exploitable pendant toute la période.

Les 70 % sont une borne du modèle, **pas un plafond réglementaire**. Ce calcul ne permet
pas de conclure à la conformité d'une annonce. Les champs source
`estimated_occupancy_l365d` et `estimated_revenue_l365d`, s'ils existent, ne sont pas écrasés.


In [ ]:
if not (0 < TAUX_AVIS <= 1 and 0 < PLAFOND_OCCUPATION <= 1):
    raise ValueError('Les taux doivent être compris entre 0 exclu et 1 inclus.')
if DUREE_SEJOUR_DEFAUT < 1:
    raise ValueError('La durée de séjour de référence doit être positive.')

df['duree_sejour_hypothese'] = df['minimum_nuits_utilisable'].clip(lower=DUREE_SEJOUR_DEFAUT)
df['sejours_estimes_avant_plafond'] = df['avis_12m_utilisables'] / TAUX_AVIS
nuits_sans_plafond = df['sejours_estimes_avant_plafond'] * df['duree_sejour_hypothese']
df['nuits_estimees_12m'] = nuits_sans_plafond.clip(upper=JOURS_ANNEE * PLAFOND_OCCUPATION)
df['estimation_plafonnee'] = nuits_sans_plafond.gt(JOURS_ANNEE * PLAFOND_OCCUPATION)
df['taux_occupation_estime_12m'] = df['nuits_estimees_12m'] / JOURS_ANNEE
df['revenu_brut_estime_12m'] = df['nuits_estimees_12m'] * df['prix_eur']

variables_b1 = [
    'prix_par_personne', 'nb_annonces_hote_local', 'hote_multi_annonce',
    'activite_avis_12m', 'taux_disponibilite_365',
    'sejours_estimes_avant_plafond', 'nuits_estimees_12m',
    'taux_occupation_estime_12m', 'revenu_brut_estime_12m'
]


### 5. Contrôler et préparer l'export
Les contrôles vérifient les bornes, l'absence de perte de lignes et le maintien des valeurs
manquantes. Le bilan porte sur les **lignes**, y compris les doublons du fichier brut.
Il ne faut donc pas additionner les revenus ou compter les annonces de cet export provisoire
pour publier un résultat. Le fichier final B1 devra avoir **une ligne par id** après A5.


In [ ]:
assert len(df) == len(source), 'Le calcul a modifié le nombre de lignes.'
assert df['taux_disponibilite_365'].dropna().between(0, 1).all()
assert df['taux_occupation_estime_12m'].dropna().between(0, PLAFOND_OCCUPATION).all()
assert df['prix_par_personne'].dropna().gt(0).all()
assert df['revenu_brut_estime_12m'].dropna().ge(0).all()
assert df.loc[df['prix_eur'].isna(), 'revenu_brut_estime_12m'].isna().all()
assert df.loc[df['capacite_utilisable'].isna(), 'prix_par_personne'].isna().all()
assert df.loc[df['minimum_nuits_utilisable'].isna(), 'nuits_estimees_12m'].isna().all()

bilan = pd.DataFrame({
    'lignes_renseignees': df[variables_b1].notna().sum(),
    'lignes_manquantes': df[variables_b1].isna().sum(),
    'pct_manquants': df[variables_b1].isna().mean().mul(100).round(2)
})
print(bilan.to_string())
print('Lignes dont l’estimation atteint le plafond :', df['estimation_plafonnee'].sum())
print('Contrôles réussis. Statut :', 'table déclarée propre' if SOURCE_NETTOYEE else 'PROVISOIRE')


In [ ]:
# Export limité aux clés, entrées utiles, variables et indicateurs de qualité.
# Aucun nom, commentaire, photographie, URL de profil ou adresse n'est exporté.
# Les clés id / host_id restent des identifiants : fichier de travail interne.
sorties = DOSSIER / 'sorties_b1'
sorties.mkdir(parents=True, exist_ok=True)
statut = 'validee' if SOURCE_NETTOYEE else 'provisoire'
sortie_csv = sorties / f'variables_b1_{statut}.csv'
df.to_csv(sortie_csv, index=False, encoding='utf-8-sig')
bilan.to_csv(sorties / f'bilan_variables_b1_{statut}.csv', encoding='utf-8-sig')
print('Variables exportées dans :', sortie_csv.resolve())
print('Pour une future jointure, lire id et host_id avec dtype="string".')

# Journal des candidats : corrections acceptées et cas restés masqués.
colonnes_audit = ['ligne_source', 'id', 'price', 'estimated_occupancy_l365d',
                  'estimated_revenue_l365d', 'prix_reconstitue_source',
                  'prix_eur', 'prix_corrige', 'prix_statut', 'prix_justification']
df.loc[candidat | prix_extreme, colonnes_audit].to_csv(
    sorties / f'audit_prix_b1_{statut}.csv', index=False, encoding='utf-8-sig'
)
print('Journal des corrections exporté dans :', sorties / f'audit_prix_b1_{statut}.csv')


### 6. conclusion
« Nous avons créé neuf variables décrivant le prix par personne, le nombre d'annonces
par hôte, le caractère multi-annonce, l'activité indiquée par les avis, la disponibilité,
ainsi que les séjours, nuits, taux d'occupation et revenus bruts estimés. Le comptage des
annonces par hôte repose sur des identifiants distincts. L'activité est un indicateur indirect
fondé sur la présence d'avis au cours des douze derniers mois.

Le modèle d'occupation suppose un taux d'avis de 50 %, une durée de séjour de trois nuits
relevée au minimum de l'annonce et une occupation maximale de 70 %. Le revenu brut est
calculé en multipliant les nuits estimées par le prix affiché. Ces paramètres constituent
des hypothèses, et les résultats ne correspondent pas à des réservations ou revenus observés.
Les prix CAD/dollars ou négatifs sont récupérés lorsque le revenu et les nuits estimés
d'origine permettent une reconstruction cohérente avec le montant initial, compte tenu des arrondis.
Cette correction repose sur une hypothèse de revenu source en euros et reste à valider en A5.
Les valeurs originales et les justifications sont conservées. Les cas incertains et les
prix supérieurs ou égaux au seuil provisoire de 9 999 € restent masqués. Les calculs présentés
sur les données brutes sont provisoires et seront réexécutés sur les tables validées. »



## B2 — Agrégation des avis et jointure contrôlée

Regrouper les avis, joindre les annonces et contrôler la jointure


Objectif : obtenir **une ligne par annonce**, enrichie du nombre d'avis retenus, des dates
du premier et du dernier avis, et du nombre d'avis sur 365 jours. Les avis mensuels
sont également préparés pour la saisonnalité.



In [ ]:
DOSSIER = SORTIES_B
DEBUT_365 = DATE_REFERENCE - pd.Timedelta(days=365)

# Tables nettoyées (A6) converties en texte : B2 conserve ainsi ses contrôles de clés et de dates.
colonnes_listings = ["id", "host_id", "neighbourhood_cleansed", "room_type", "price", "accommodates",
                     "minimum_nights", "availability_365", "number_of_reviews", "number_of_reviews_ltm",
                     "last_review"]
listings_brut = listings_propre[colonnes_listings].copy()
listings_brut["last_review"] = listings_brut["last_review"].dt.strftime("%Y-%m-%d")
listings_brut = listings_brut.astype("string")
listings_brut.insert(2, "last_scraped", DATE_REFERENCE.strftime("%Y-%m-%d"))

# Variables de B1 jointes sur l'identifiant, pour qu'elles suivent dans les étapes suivantes.
variables_a_joindre = [c for c in ["prix_eur", "prix_par_personne", "nb_annonces_hote_local",
    "hote_multi_annonce", "activite_avis_12m", "taux_disponibilite_365", "sejours_estimes_avant_plafond",
    "nuits_estimees_12m", "taux_occupation_estime_12m", "revenu_brut_estime_12m"] if c in df.columns]
listings_brut = listings_brut.merge(df[["id"] + variables_a_joindre], on="id", how="left", validate="one_to_one")

reviews_brut = reviews_propre[["id", "listing_id", "date"]].copy()
reviews_brut["date"] = reviews_brut["date"].dt.strftime("%Y-%m-%d")
reviews_brut = reviews_brut.astype("string")

print("Annonces :", len(listings_brut), "| Avis :", len(reviews_brut))
print("Dates de collecte listings :", listings_brut["last_scraped"].value_counts(dropna=False).to_dict())
print("Fenêtre :", DEBUT_365.date(), "exclu →", DATE_REFERENCE.date(), "inclus")

### 1. Traiter les clés avant la jointure 
On supprime uniquement les **doublons exacts sur toutes les colonnes** dans des copies.
Si un même identifiant reste présent plusieurs fois, toutes ses versions sont mises à part.
Il peut s'agir d'une contradiction ou d'une différence de format à résoudre en A5.
On ne garde pas automatiquement la première ligne.

Les identifiants absents ou non valides sont aussi mis à part. Le journal conserve les
numéros de lignes de données (hors en-tête) pour retrouver les enregistrements sources.
Les fichiers bruts ne sont jamais écrasés.


In [ ]:
doublons_listings = listings_brut.duplicated()
doublons_reviews = reviews_brut.duplicated()
listings = listings_brut.loc[~doublons_listings].copy()
reviews = reviews_brut.loc[~doublons_reviews].copy()

id_l_valide = listings['id'].str.fullmatch(r'[1-9][0-9]*', na=False)
id_r_valide = reviews['id'].str.fullmatch(r'[1-9][0-9]*', na=False)
cle_r_valide = reviews['listing_id'].str.fullmatch(r'[1-9][0-9]*', na=False)
conflit_l = listings['id'].duplicated(keep=False)
conflit_r = reviews['id'].duplicated(keep=False)

listings_uniques = listings.loc[id_l_valide & ~conflit_l].copy()
assert listings_uniques['id'].is_unique

print('Doublons exacts retirés :', int(doublons_listings.sum()), 'annonces ;', int(doublons_reviews.sum()), 'avis')
print('Identifiants listings répétés après retrait des doublons exacts :', listings.loc[conflit_l, 'id'].nunique())
print('Identifiants reviews répétés après retrait des doublons exacts :', reviews.loc[conflit_r, 'id'].nunique())
print('Lignes listings mises à part :', int((~id_l_valide | conflit_l).sum()))
print('Annonces uniques conservées :', len(listings_uniques))

if SOURCE_NETTOYEE and (doublons_listings.any() or doublons_reviews.any()
                       or conflit_l.any() or conflit_r.any()
                       or (~id_l_valide).any() or (~id_r_valide).any() or (~cle_r_valide).any()):
    raise ValueError('Les tables déclarées propres ont encore des problèmes de clés ou des doublons : revoir A5.')


### 2. Vérifier les dates et repérer les avis hors périmètre 
Le snapshot annoncé est le **17/09/2026**, mais `last_scraped` indique le **19/09/2026**
dans les annonces fournies. Nous retenons explicitement le **19/09/2026** pour rester
cohérents avec B1 et ne pas exclure des avis déjà collectés le 18 ou le 19.

Formats acceptés : `AAAA-MM-JJ` et `JJ/MM/AAAA`, sans deviner les dates ambiguës.
Les dates impossibles, absentes ou postérieures à la date de référence sont exclues
des agrégations, avec une trace dans le journal. Les 365 jours couvrent
**(date de référence − 365 jours, date de référence]**.

On distingue les avis d'une annonce absente du fichier source et ceux d'une annonce
présente mais écartée pour conflit. Ces avis ne sont pas rattachés à une autre annonce.


In [ ]:
date_texte = reviews['date'].str.strip()
date_iso = pd.to_datetime(date_texte, format='%Y-%m-%d', errors='coerce')
date_fr = pd.to_datetime(date_texte, format='%d/%m/%Y', errors='coerce')
reviews['date_avis'] = date_iso.fillna(date_fr)
date_invalide = reviews['date_avis'].isna()
date_future = reviews['date_avis'].gt(DATE_REFERENCE)

avis_utilisable = id_r_valide & cle_r_valide & ~conflit_r & ~date_invalide & ~date_future
reviews_utilisables = reviews.loc[avis_utilisable].copy()
reviews_utilisables['dans_365j'] = reviews_utilisables['date_avis'].gt(DEBUT_365)

ids_source = listings_brut['id'].dropna().unique()
ids_conserves = listings_uniques['id']
annonce_absente = ~reviews['listing_id'].isin(ids_source)
annonce_ecartee = reviews['listing_id'].isin(ids_source) & ~reviews['listing_id'].isin(ids_conserves)

# Plusieurs indicateurs peuvent concerner le même avis : ne pas additionner ces motifs.
audit_reviews = reviews[['id', 'listing_id', 'date']].copy()
audit_reviews.insert(0, 'ligne_source', reviews.index + 1)
audit_reviews['id_invalide'] = ~id_r_valide
audit_reviews['listing_id_invalide'] = ~cle_r_valide
audit_reviews['id_repete_apres_dedoublonnage'] = conflit_r
audit_reviews['date_invalide'] = date_invalide
audit_reviews['date_future'] = date_future
audit_reviews['annonce_absente_source'] = annonce_absente
audit_reviews['annonce_ecartee'] = annonce_ecartee
audit_reviews['retenu_pour_jointure'] = avis_utilisable & ~annonce_absente & ~annonce_ecartee

print('Dates converties depuis le format français :', int((date_iso.isna() & date_fr.notna()).sum()))
print('Dates invalides :', int(date_invalide.sum()), '| Dates futures :', int(date_future.sum()))
print('Avis utilisables avant sélection des annonces :', len(reviews_utilisables))
print('Avis utilisables sans annonce source :', int((avis_utilisable & annonce_absente).sum()))
print('Avis utilisables dont l’annonce a été écartée :', int((avis_utilisable & annonce_ecartee).sum()))

if SOURCE_NETTOYEE and (date_invalide.any() or date_future.any()):
    raise ValueError('Les dates des avis déclarés propres restent invalides ou dépassent la date choisie.')


### 3. Regrouper les avis par annonce
`groupby('listing_id')` rassemble les avis d'une même annonce. `agg` produit :
- `nb_avis_retenus` : nombre total d'avis retenus jusqu'à la date de référence ;
- `premier_avis_retenu`, `dernier_avis_retenu` : dates extrêmes de ces avis ;
- `nb_avis_365j` : nombre d'avis retenus sur la fenêtre de 365 jours.

Le tableau agrégé possède **une seule ligne par annonce**. Cela évite qu'une annonce
avec 100 avis soit répétée 100 fois dans la jointure. Le tableau mensuel est séparé,
car il possède plusieurs lignes par annonce. Il mesure les avis, pas les réservations.


In [ ]:
avis_par_annonce = (
    reviews_utilisables.groupby('listing_id', as_index=False)
    .agg(nb_avis_retenus=('id', 'count'),
         premier_avis_retenu=('date_avis', 'min'),
         dernier_avis_retenu=('date_avis', 'max'),
         nb_avis_365j=('dans_365j', 'sum'))
)
assert avis_par_annonce['listing_id'].is_unique

# Saison : ne garder que les annonces retenues, sans fusionner ce tableau à listings.
r_mensuels = reviews_utilisables.loc[reviews_utilisables['listing_id'].isin(ids_conserves)].copy()
r_mensuels['mois'] = r_mensuels['date_avis'].dt.to_period('M').astype('string')
avis_mensuels = (r_mensuels.groupby(['listing_id', 'mois'], as_index=False)
                 .agg(nb_avis=('id', 'count')))
print('Annonces présentes dans l’agrégation des avis :', len(avis_par_annonce))
print('Lignes annonce × mois :', len(avis_mensuels))
print('Somme des avis agrégés :', int(avis_par_annonce['nb_avis_retenus'].sum()))


### 4. Joindre les avis aux annonces 
On effectue une **jointure gauche** : toutes les annonces retenues restent présentes,
y compris celles sans avis retenu. La clé est `listings.id = reviews.listing_id`.
`validate='one_to_one'` bloque la jointure si une clé est répétée dans l'une des tables.

Les compteurs absents deviennent zéro : cela signifie **zéro avis retenu dans notre
périmètre**, pas nécessairement aucune réservation ou aucun avis dans les données brutes.
Les dates restent manquantes. `avis_jointure` conserve l'indication de correspondance.
Les noms de voyageurs, commentaires et profils ne sont pas exportés.


In [ ]:
# Colonnes utiles à la suite ; ajouter au besoin des colonnes validées avec le membre 1.
colonnes_utiles = [
    'id', 'host_id', 'last_scraped', 'neighbourhood_cleansed', 'room_type',
    'price', 'accommodates', 'minimum_nights', 'availability_365',
    'number_of_reviews', 'number_of_reviews_ltm', 'last_review',
    'prix_eur', 'prix_par_personne', 'nb_annonces_hote_local', 'hote_multi_annonce',
    'activite_avis_12m', 'taux_disponibilite_365', 'sejours_estimes_avant_plafond',
    'nuits_estimees_12m', 'taux_occupation_estime_12m', 'revenu_brut_estime_12m'
]
colonnes_presentes = [c for c in colonnes_utiles if c in listings_uniques.columns]
base = listings_uniques[colonnes_presentes].copy()
n_avant = len(base)
listings_avec_avis = base.merge(
    avis_par_annonce, how='left', left_on='id', right_on='listing_id',
    validate='one_to_one', indicator='avis_jointure'
).drop(columns='listing_id')

compteurs = ['nb_avis_retenus', 'nb_avis_365j']
listings_avec_avis[compteurs] = listings_avec_avis[compteurs].fillna(0).astype('Int64')

# Compter séparément les avis écartés après retrait des doublons exacts.
ecartes_par_annonce = reviews.loc[~avis_utilisable].groupby('listing_id').size()
listings_avec_avis['nb_lignes_avis_ecartees'] = (
    listings_avec_avis['id'].map(ecartes_par_annonce).fillna(0).astype('Int64')
)
print('Lignes avant / après jointure :', n_avant, '/', len(listings_avec_avis))
print(listings_avec_avis['avis_jointure'].value_counts().to_string())


### 5. Contrôler la jointure et les écarts 
Les contrôles portent sur le nombre et l'ensemble des identifiants, l'unicité, la
conservation des avis rattachables, les dates et la cohérence des compteurs.
On compare aussi les agrégats aux compteurs `number_of_reviews` et `number_of_reviews_ltm`.
Un écart est un **diagnostic** : il peut venir du nettoyage, de la date de référence,
de la collecte ou d'une altération pédagogique. On ne force pas les valeurs à être égales.
Les compteurs source ne sont pas écrasés.


In [ ]:
assert len(listings_avec_avis) == n_avant, 'La jointure a multiplié ou perdu des lignes.'
assert listings_avec_avis['id'].is_unique
assert set(listings_avec_avis['id']) == set(base['id'])
assert listings_avec_avis['id'].notna().all()
assert listings_avec_avis[compteurs].ge(0).all().all()
assert listings_avec_avis['nb_avis_365j'].le(listings_avec_avis['nb_avis_retenus']).all()
assert int(listings_avec_avis['nb_avis_retenus'].sum()) == len(r_mensuels)
assert int(avis_mensuels['nb_avis'].sum()) == len(r_mensuels)
assert int(listings_avec_avis['nb_avis_365j'].sum()) == int(r_mensuels['dans_365j'].sum())
avec_avis = listings_avec_avis['nb_avis_retenus'].gt(0)
assert listings_avec_avis.loc[avec_avis, 'premier_avis_retenu'].le(
    listings_avec_avis.loc[avec_avis, 'dernier_avis_retenu']).all()
assert listings_avec_avis.loc[avec_avis, 'dernier_avis_retenu'].le(DATE_REFERENCE).all()
assert listings_avec_avis.loc[~avec_avis, ['premier_avis_retenu', 'dernier_avis_retenu']].isna().all().all()

for original, calcule, ecart in [
    ('number_of_reviews', 'nb_avis_retenus', 'ecart_avis_total'),
    ('number_of_reviews_ltm', 'nb_avis_365j', 'ecart_avis_365j')
]:
    valeur_source = pd.to_numeric(listings_avec_avis[original], errors='coerce')
    valeur_source = valeur_source.where(valeur_source.ge(0) & valeur_source.mod(1).eq(0))
    listings_avec_avis[ecart] = listings_avec_avis[calcule] - valeur_source
    print(ecart, ':', int(listings_avec_avis[ecart].ne(0).fillna(False).sum()),
          'annonces en désaccord ;', int(listings_avec_avis[ecart].isna().sum()), 'non comparables')

bilan = pd.DataFrame([
    ('listings_lignes_brutes', len(listings_brut)),
    ('listings_doublons_exacts_retires', int(doublons_listings.sum())),
    ('listings_lignes_ecartees_apres_dedoublonnage', len(listings)-len(base)),
    ('listings_lignes_avant_jointure', n_avant),
    ('listings_lignes_apres_jointure', len(listings_avec_avis)),
    ('annonces_avec_avis_retenus', int(avec_avis.sum())),
    ('annonces_sans_avis_retenu', int((~avec_avis).sum())),
    ('reviews_lignes_brutes', len(reviews_brut)),
    ('reviews_doublons_exacts_retires', int(doublons_reviews.sum())),
    ('reviews_ecartees_cles_ou_dates', int((~avis_utilisable).sum())),
    ('reviews_utilisables_annonce_absente', int((avis_utilisable & annonce_absente).sum())),
    ('reviews_utilisables_annonce_ecartee', int((avis_utilisable & annonce_ecartee).sum())),
    ('reviews_retenues_jointure', len(r_mensuels)),
    ('reviews_retenues_365j', int(r_mensuels['dans_365j'].sum()))
], columns=['controle', 'valeur'])
assert len(reviews_brut) == (int(doublons_reviews.sum()) + int((~avis_utilisable).sum())
    + int((avis_utilisable & annonce_absente).sum())
    + int((avis_utilisable & annonce_ecartee).sum()) + len(r_mensuels))
print(bilan.to_string(index=False))
print('Contrôles de jointure réussis.')


### 6. Exporter les résultats et les journaux 
Les fichiers sont créés dans `sorties_b2_jointure` : table des annonces enrichies,
avis agrégés par annonce, avis mensuels, bilan, écarts avec les compteurs source et
journaux des lignes écartées. Les journaux gardent les clés et motifs, sans textes d'avis.
Les identifiants restent des clés de travail internes, pas des données anonymes à publier.

Les prix de la table source ne sont pas nettoyés ici. Si l'entrée contient déjà des
variables B1, elles sont conservées parmi les colonnes prévues ; elles ne sont pas
recalculées automatiquement avec les nouveaux compteurs d'avis.


In [ ]:
sorties = DOSSIER / 'sorties_b2_jointure'
sorties.mkdir(parents=True, exist_ok=True)
statut = 'validee' if SOURCE_NETTOYEE else 'provisoire'
listings_avec_avis.to_csv(sorties / f'listings_avec_avis_{statut}.csv', index=False, encoding='utf-8-sig')
avis_par_annonce.loc[avis_par_annonce['listing_id'].isin(ids_conserves)].to_csv(
    sorties / f'avis_par_annonce_{statut}.csv', index=False, encoding='utf-8-sig')
avis_mensuels.to_csv(sorties / f'avis_mensuels_{statut}.csv', index=False, encoding='utf-8-sig')
bilan.to_csv(sorties / f'bilan_jointure_{statut}.csv', index=False, encoding='utf-8-sig')
audit_reviews.loc[~audit_reviews['retenu_pour_jointure']].to_csv(
    sorties / f'avis_ecartes_{statut}.csv', index=False, encoding='utf-8-sig')

# Journal des doublons exacts : les fichiers source permettent de retrouver le contenu.
for brut, masque, nom, cles in [
    (listings_brut, doublons_listings, 'listings', ['id']),
    (reviews_brut, doublons_reviews, 'reviews', ['id', 'listing_id'])
]:
    journal = brut.loc[masque, cles].copy()
    journal.insert(0, 'ligne_source', journal.index + 1)
    journal.to_csv(sorties / f'doublons_exacts_{nom}_{statut}.csv', index=False, encoding='utf-8-sig')

journal_l = listings.loc[~id_l_valide | conflit_l, ['id']].copy()
journal_l.insert(0, 'ligne_source', journal_l.index + 1)
journal_l['id_invalide'] = ~id_l_valide.loc[journal_l.index]
journal_l['id_repete'] = conflit_l.loc[journal_l.index]
journal_l.to_csv(sorties / f'listings_ecartes_{statut}.csv', index=False, encoding='utf-8-sig')
colonnes_ecart = ['id', 'number_of_reviews', 'nb_avis_retenus', 'ecart_avis_total',
                  'number_of_reviews_ltm', 'nb_avis_365j', 'ecart_avis_365j', 'nb_lignes_avis_ecartees']
listings_avec_avis[colonnes_ecart].to_csv(sorties / f'comparaison_compteurs_{statut}.csv', index=False, encoding='utf-8-sig')
print('Exports créés dans :', sorties.resolve())
print('Lors de la relecture des CSV, garder id et listing_id en texte (dtype="string").')


### Conclusion 
« Les avis ont été regroupés par annonce afin de calculer leur nombre total retenu,
leur nombre sur 365 jours et les dates du premier et du dernier avis. Ces indicateurs
ont été joints aux annonces par une jointure gauche contrôlée de type un-à-un.
Le nombre et les identifiants des annonces retenues ont été conservés.
Les annonces sans avis retenu disposent de compteurs nuls et de dates manquantes.

Les doublons exacts ont été retirés des copies de travail. Les clés contradictoires,
les dates invalides ou futures et les avis non rattachables ont été consignés dans
des journaux distincts. Les écarts avec les compteurs d'origine ont été mesurés sans
écraser ces derniers. Les résultats restent provisoires jusqu'à validation du nettoyage. »



### Source externe : population de Lyon par arrondissement

**Source :** Insee, *Populations de référence en vigueur à compter du 1er janvier 2026* (date de référence statistique : 1er janvier 2023), authentifiées par le décret n° 2025-1362 du 26 décembre 2025. Fichier « 69 Rhône », tableau 3 : https://www.insee.fr/fr/statistiques/fichier/8680694/dep69.pdf (consulté le 21/11/2026). On retient la **population municipale** (résidents habituels), saisie dans `docs/sources/population_lyon.csv`.

**Limites :** la population résidente ne mesure ni la demande touristique ni le nombre de logements disponibles. Les arrondissements centraux (1er, 2e), peu peuplés, voient leur densité d'annonces gonflée mécaniquement.

In [ ]:
population = pd.read_csv(PROJET / "docs" / "sources" / "population_lyon.csv")

# Contrôles : mêmes noms que les annonces, et total égal à celui publié par l'Insee pour la commune de Lyon
arrondissements_annonces = set(listings_propre["neighbourhood_cleansed"].unique())
absents = arrondissements_annonces - set(population["arrondissement"])
assert not absents, f"Arrondissements sans population : {absents}"
assert population["population_municipale"].sum() == 519127, "Total différent de la population de Lyon (Insee)"
print("Population municipale de Lyon :", f"{population['population_municipale'].sum():,}".replace(",", " "), "habitants")
population

## B3 — Q1 : annonces actives et saisonnalité

Annonces actives et saisonnalité mensuelle à Lyon

**Définition choisie :** une annonce est active selon les avis si elle possède au moins
un avis retenu dans les 365 jours précédant la date de référence, borne finale incluse.
C'est un indicateur indirect d'activité : une annonce sans avis récent n'est pas forcément inactive.

Le dénominateur est l'ensemble des annonces retenues en B2, pas uniquement celles avec un prix
ou des avis. Le calendrier disponible ne sert pas à prouver une location.



In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

DOSSIER_B2 = SORTIES_B / 'sorties_b2_jointure'
SORTIES = SORTIES_B / 'sorties_b3_q1'
STATUT = 'validee'
DATE_REFERENCE = pd.Timestamp('2026-09-19')  # Même référence que B2.

f_annonces = DOSSIER_B2 / f'listings_avec_avis_{STATUT}.csv'
f_mois = DOSSIER_B2 / f'avis_mensuels_{STATUT}.csv'
for f in [f_annonces, f_mois]:
    if not f.is_file():
        raise FileNotFoundError(f'Absent : {f.resolve()} ; exécuter B2 ou modifier DOSSIER_B2.')

annonces = pd.read_csv(f_annonces, dtype={'id': 'string', 'host_id': 'string'})
avis_mois = pd.read_csv(f_mois, dtype={'listing_id': 'string', 'mois': 'string'})
for colonne in ['premier_avis_retenu', 'dernier_avis_retenu']:
    annonces[colonne] = pd.to_datetime(annonces[colonne], errors='raise')
for colonne in ['nb_avis_365j', 'nb_avis_retenus']:
    annonces[colonne] = pd.to_numeric(annonces[colonne], errors='raise')
avis_mois['nb_avis'] = pd.to_numeric(avis_mois['nb_avis'], errors='raise')
avis_mois['periode'] = pd.PeriodIndex(avis_mois['mois'], freq='M')

assert len(annonces) > 0 and annonces['id'].notna().all() and annonces['id'].is_unique
assert annonces[['nb_avis_365j', 'nb_avis_retenus']].notna().all().all()
assert annonces[['nb_avis_365j', 'nb_avis_retenus']].ge(0).all().all()
assert annonces[['nb_avis_365j', 'nb_avis_retenus']].mod(1).eq(0).all().all()
assert annonces['nb_avis_365j'].le(annonces['nb_avis_retenus']).all()
assert avis_mois['nb_avis'].notna().all() and avis_mois['nb_avis'].gt(0).all()
assert avis_mois['nb_avis'].mod(1).eq(0).all()
assert not avis_mois.duplicated(['listing_id', 'periode']).any()
assert avis_mois['listing_id'].isin(annonces['id']).all()
assert avis_mois['periode'].le(DATE_REFERENCE.to_period('M')).all()
assert avis_mois['nb_avis'].sum() == annonces['nb_avis_retenus'].sum()
assert annonces['dernier_avis_retenu'].dropna().le(DATE_REFERENCE).all()
print('Entrées contrôlées :', len(annonces), 'annonces ;', len(avis_mois), 'lignes annonce × mois.')


### 1. Mesurer les annonces actives
`active_365j = nb_avis_365j > 0`. Le taux est le nombre d'annonces actives divisé par
le nombre total d'annonces retenues, multiplié par 100.

La fenêtre choisie est **du 20 septembre 2025 au 19 septembre 2026 inclus**.
Le 19/09 correspond au `last_scraped` observé ; le snapshot annoncé était le 17/09.


In [ ]:
debut_365 = DATE_REFERENCE - pd.Timedelta(days=365)
annonces['active_365j'] = annonces['nb_avis_365j'].gt(0)
active_selon_date = annonces['dernier_avis_retenu'].gt(debut_365)
assert annonces['active_365j'].equals(active_selon_date), 'Compteurs/dates incompatibles : vérifier B2 et sa référence.'

n_total = len(annonces)
n_actives = int(annonces['active_365j'].sum())
taux_actives = 100 * n_actives / n_total
bilan_activite = pd.DataFrame({
    'categorie': ['Actives selon les avis', 'Sans avis récent retenu'],
    'nb_annonces': [n_actives, n_total - n_actives]
})
bilan_activite['pourcentage'] = bilan_activite['nb_annonces'] / n_total * 100
print(bilan_activite.round(2).to_string(index=False))

fig_activite = px.bar(
    bilan_activite, x='categorie', y='nb_annonces', color='categorie', text='nb_annonces',
    color_discrete_sequence=['#087F8C', '#9CA3AF'],
    title=f'Activité selon les avis — {taux_actives:.1f} % des annonces retenues',
    labels={'categorie': '', 'nb_annonces': 'Nombre d’annonces', 'pourcentage': 'Part (%)'},
    hover_data={'pourcentage': ':.1f'}, template='plotly_white'
)
fig_activite.update_layout(showlegend=False)
fig_activite.show()


### 1 bis. Sensibilité à la définition d’activité

On compare la définition retenue à deux variantes plus restrictives.

| Définition | Règle | Interprétation |
|---|---|---|
| Retenue | Au moins un avis retenu sur 365 jours | Signe d’activité récente, sans preuve directe de réservation. |
| Variante 1 | Dernier avis depuis six mois, borne incluse | Retient les avis plus récents. L’absence d’avis ne prouve pas un arrêt d’activité. |
| Variante 2 | Avis sur 365 jours et `availability_365 > 0` | Ajoute une disponibilité future déclarée. Un calendrier sans disponibilité peut être complet ou bloqué ; cette règle ne prouve pas qu’un logement est réellement réservable. |




In [ ]:
# Sensibilité du chiffre clé à la définition
import matplotlib.pyplot as plt


def nombre_fr(x):
    # 4573 -> '4 573'
    return f'{x:,.0f}'.replace(',', ' ')


def pct_fr(x, decimales=1):
    # 49.84 -> '49,8 %'
    return f'{x:.{decimales}f}'.replace('.', ',') + ' %'


debut_6m = DATE_REFERENCE - pd.DateOffset(months=6)
disponibilite = pd.to_numeric(annonces['availability_365'], errors='coerce')
definitions = {
    'Retenue : au moins 1 avis sur 365 jours': annonces['active_365j'],
    'Variante : avis depuis 6 mois (borne incluse)': annonces['dernier_avis_retenu'].ge(debut_6m),
    'Variante : 1 avis sur 365 jours et calendrier ouvert': annonces['active_365j'] & disponibilite.gt(0),
}
sensibilite = pd.DataFrame([
    {'definition': nom, 'nb_annonces': int(masque.sum()), 'part_pct': 100 * masque.mean()}
    for nom, masque in definitions.items()
])
variantes = sensibilite.iloc[1:]
ecarts = 100 * (variantes['nb_annonces'] / n_actives - 1)
assert sensibilite['nb_annonces'].le(n_total).all()
print(sensibilite.round(1).to_string(index=False))
print(f"Écart des variantes par rapport à la définition retenue : de {ecarts.min():.0f} % à {ecarts.max():.0f} %")

fig_sensibilite, ax = plt.subplots(figsize=(8, 3.2))
ordre = sensibilite.iloc[::-1]
couleurs = ['#087F8C' if d.startswith('Retenue') else '#9CA3AF' for d in ordre['definition']]
barres = ax.barh(ordre['definition'], ordre['nb_annonces'], color=couleurs)
for barre, part in zip(barres, ordre['part_pct']):
    ax.text(barre.get_width() + 50, barre.get_y() + barre.get_height() / 2,
            f'{nombre_fr(barre.get_width())} ({pct_fr(part, 0)})', va='center', fontsize=9)
ax.set_title(f'{nombre_fr(n_actives)} annonces actives selon les avis sur {nombre_fr(n_total)} ({pct_fr(taux_actives, 0)}) ;\n'
             f'de {nombre_fr(variantes["nb_annonces"].min())} à {nombre_fr(variantes["nb_annonces"].max())} '
             'avec une définition plus stricte', loc='left', fontsize=11)
ax.set_xlabel("Nombre d'annonces selon la définition")
ax.set_xlim(0, sensibilite['nb_annonces'].max() * 1.25)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: nombre_fr(x)))
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

### 2. Construire la série mensuelle sur 24 mois complets
Pour chaque mois, on calcule le **nombre d'avis**, le **nombre d'annonces ayant reçu un avis**
et les **avis par jour calendaire**. L'ensemble des annonces retenues en B2 sert de périmètre
historique ; on ne filtre pas la série sur les seules annonces actives aujourd'hui.

Le mois de la collecte est isolé s'il est incomplet. Ici, septembre 2026 couvre seulement
les 19 premiers jours : il ne permet pas de conclure à une baisse par rapport à août.
Les mois complets retenus vont de septembre 2024 à août 2026.
Les mois sans avis sont explicitement complétés par zéro dans cette cohorte observée.


In [ ]:
mois_reference = DATE_REFERENCE.to_period('M')
mois_incomplet = DATE_REFERENCE.day < DATE_REFERENCE.days_in_month
dernier_complet = mois_reference - 1 if mois_incomplet else mois_reference
periodes = pd.period_range(end=dernier_complet, periods=24, freq='M')

serie = avis_mois.groupby('periode').agg(
    nb_avis=('nb_avis', 'sum'),
    nb_annonces_avec_avis=('listing_id', 'nunique')
)
mensuel = serie.reindex(periodes, fill_value=0)
mensuel.index.name = 'periode'
mensuel = mensuel.reset_index()
mensuel['mois'] = mensuel['periode'].astype(str)
mensuel['date_mois'] = mensuel['periode'].dt.to_timestamp()
mensuel['jours'] = mensuel['periode'].dt.days_in_month
mensuel['avis_par_jour'] = mensuel['nb_avis'] / mensuel['jours']
mensuel['part_annonces_avec_avis_pct'] = 100 * mensuel['nb_annonces_avec_avis'] / n_total
assert len(mensuel) == 24 and mensuel['nb_annonces_avec_avis'].le(n_total).all()
assert mensuel['nb_avis'].sum() == avis_mois.loc[avis_mois['periode'].isin(periodes), 'nb_avis'].sum()

partiel = serie.reindex([mois_reference], fill_value=0).copy() if mois_incomplet else serie.iloc[0:0].copy()
if mois_incomplet:
    partiel['jours_observes'] = DATE_REFERENCE.day
    partiel['mois_incomplet'] = True
    print('Mois partiel, exclu des comparaisons :')
    print(partiel.to_string())
print(mensuel[['mois', 'nb_avis', 'nb_annonces_avec_avis', 'avis_par_jour']].round(2).to_string(index=False))

fig_volume = px.line(
    mensuel, x='date_mois', y='nb_avis', markers=True, template='plotly_white',
    title='Avis mensuels — 24 mois complets, cohorte des annonces retenues',
    labels={'date_mois': 'Mois', 'nb_avis': 'Avis retenus',
            'nb_annonces_avec_avis': 'Annonces avec avis', 'avis_par_jour': 'Avis par jour'},
    hover_data={'nb_annonces_avec_avis': True, 'avis_par_jour': ':.1f'}
)
fig_volume.update_traces(line_color='#087F8C')
fig_volume.update_xaxes(dtick='M2', tickformat='%b %Y')
fig_volume.show()

fig_annonces = px.line(
    mensuel, x='date_mois', y='nb_annonces_avec_avis', markers=True, template='plotly_white',
    title='Annonces ayant reçu au moins un avis dans chaque mois complet',
    labels={'date_mois': 'Mois', 'nb_annonces_avec_avis': 'Nombre d’annonces'},
    hover_data={'part_annonces_avec_avis_pct': ':.1f'}
)
fig_annonces.update_traces(line_color='#C16E32')
fig_annonces.update_xaxes(dtick='M2', tickformat='%b %Y')
fig_annonces.show()


### 3. Distinguer variation mensuelle et saisonnalité
Un pic sur un seul mois ne prouve pas une saisonnalité. On compare **deux cycles de douze
mois complets** pour voir si les mêmes mois sont régulièrement hauts ou bas.

Pour tenir compte de la longueur des mois et du niveau différent des deux cycles :
**indice mensuel = (avis par jour du mois / avis par jour de son cycle) × 100**.
Le niveau quotidien du cycle est son total d'avis divisé par son total de jours.
Un indice de 120 signifie une activité d'avis quotidienne 20 % au-dessus du niveau du cycle.
Ce n'est pas un taux d'occupation. La normalisation réduit l'effet du niveau global,
sans supprimer une tendance à l'intérieur de chaque cycle ni les changements de composition.


In [ ]:
cycle_1 = f'{periodes[0]} à {periodes[11]}'
cycle_2 = f'{periodes[12]} à {periodes[23]}'
mensuel['cycle'] = [cycle_1] * 12 + [cycle_2] * 12
cycles = mensuel.groupby('cycle', sort=False).agg(nb_avis=('nb_avis', 'sum'), jours=('jours', 'sum'))
cycles['avis_par_jour'] = cycles['nb_avis'] / cycles['jours']
if cycles['nb_avis'].le(0).any():
    raise ValueError('Un cycle est sans avis : choisir une fenêtre historique couverte.')
mensuel['indice_saisonnier'] = 100 * mensuel['avis_par_jour'] / mensuel['cycle'].map(cycles['avis_par_jour'])
mensuel['numero_mois'] = mensuel['periode'].dt.month
noms_mois = {1:'Janvier', 2:'Février', 3:'Mars', 4:'Avril', 5:'Mai', 6:'Juin',
             7:'Juillet', 8:'Août', 9:'Septembre', 10:'Octobre', 11:'Novembre', 12:'Décembre'}
mensuel['mois_calendaire'] = mensuel['numero_mois'].map(noms_mois)
profil = mensuel.groupby('numero_mois').agg(
    indice_moyen=('indice_saisonnier', 'mean'),
    indice_min=('indice_saisonnier', 'min'),
    indice_max=('indice_saisonnier', 'max'),
    nb_observations=('indice_saisonnier', 'count')
)
profil['mois'] = profil.index.map(noms_mois)
assert profil['nb_observations'].eq(2).all()
fig_saison = px.line(
    mensuel.sort_values(['cycle', 'numero_mois']), x='mois_calendaire', y='indice_saisonnier',
    color='cycle', markers=True, template='plotly_white',
    category_orders={'mois_calendaire': list(noms_mois.values())},
    title=(f"{profil.loc[profil['indice_moyen'].idxmax(), 'mois']} a l’indice d’avis moyen le plus élevé ; "
           f"{profil.loc[profil['indice_moyen'].idxmin(), 'mois'].lower()} le plus faible (base 100 = jour moyen du cycle)"),
    labels={'mois_calendaire': '', 'indice_saisonnier': 'Indice quotidien (base 100)', 'cycle': 'Cycle'},
    color_discrete_sequence=['#087F8C', '#C16E32'], hover_data={'nb_avis': True, 'avis_par_jour': ':.1f'}
)
fig_saison.add_hline(y=100, line_dash='dash', line_color='#777777')
fig_saison.show()
print(cycles.round(2).to_string())
print(profil[['mois', 'indice_moyen', 'indice_min', 'indice_max']].round(1).to_string(index=False))


### 3 bis. Profil mensuel en Matplotlib

Les barres montrent la moyenne des indices des deux cycles ; les traits indiquent leur minimum et leur maximum. **Ce ne sont pas des intervalles de confiance.** Un mois coloré dépasse 100 dans les deux cycles observés, sans que cela démontre une saisonnalité stable.

Cette figure et la courbe Plotly représentent la même analyse : changer d’outil ne suffit pas à en faire deux analyses distinctes pour B4.


In [ ]:
# AJOUT — Profil saisonnier en Matplotlib (titre = constat)
mois_courts = ['janv.', 'févr.', 'mars', 'avr.', 'mai', 'juin', 'juil.', 'août', 'sept.', 'oct.', 'nov.', 'déc.']
profil = profil.sort_index()
mois_forts = profil['indice_min'].gt(100)
mois_haut = profil.loc[profil['indice_moyen'].idxmax()]
mois_bas = profil.loc[profil['indice_moyen'].idxmin()]

fig_profil, ax = plt.subplots(figsize=(8.5, 4))
ax.bar(mois_courts, profil['indice_moyen'], color=['#087F8C' if f else '#9CA3AF' for f in mois_forts])
ax.errorbar(mois_courts, profil['indice_moyen'],
            yerr=[profil['indice_moyen'] - profil['indice_min'], profil['indice_max'] - profil['indice_moyen']],
            fmt='none', ecolor='#333333', capsize=3, linewidth=1)
ax.axhline(100, color='#777777', linestyle='--', linewidth=1)
ax.text(-0.4, 102, 'jour moyen = 100', ha='left', va='bottom', fontsize=8, color='#555555')
ax.set_title(f"{mois_haut['mois']} dépasse un jour moyen de {mois_haut['indice_moyen'] - 100:.0f} % ; "
             f"{mois_bas['mois'].lower()} est {100 - mois_bas['indice_moyen']:.0f} % en dessous\n"
             f"(moyenne de deux cycles, {cycle_1} et {cycle_2})", loc='left', fontsize=11)
ax.set_xlabel('Mois')
ax.set_ylabel('Avis par jour, indice (100 = jour moyen)')
ax.set_ylim(0, profil['indice_max'].max() * 1.12)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()
print('Barres : moyenne des deux cycles ; traits : minimum et maximum ; en couleur : au-dessus de 100 dans les deux cycles.')

### 4. Résumer les résultats et leurs limites
Les conclusions ci-dessous sont calculées automatiquement à partir des fichiers B2.
Elles portent sur les **avis publiés**, dont la date peut être décalée par rapport au séjour.
Les données ne permettent pas d'attribuer un pic à une fête, au tourisme ou à un événement
sans une recherche complémentaire. Deux cycles constituent une indication descriptive,
pas une démonstration d'une saisonnalité stable ni une prévision.

Le stock d'annonces est celui retenu au moment de la collecte : les annonces disparues
ne sont pas représentées. Les exclusions et corrections pédagogiques peuvent modifier
les résultats. « Sans avis récent retenu » ne doit pas être renommé « inactif certain ».


### Lecture pour l’investisseur

La conclusion décrit la part d’annonces avec un signe d’activité récente et les mois où les avis sont relativement nombreux. **Elle ne mesure pas les revenus mensuels**, car les prix sont observés à une seule date.

Une éventuelle tendance dans le temps peut influencer le profil mensuel. La hausse entre les deux cycles ne prouve pas une hausse à l’intérieur de chaque cycle. Les pistes concernant l’entretien doivent être vérifiées avec les réservations réelles.


In [ ]:
recent = mensuel.loc[mensuel['cycle'].eq(cycle_2)]
pic = recent.loc[recent['nb_avis'].idxmax()]
creux = recent.loc[recent['nb_avis'].idxmin()]
variation = 100 * (cycles.loc[cycle_2, 'nb_avis'] / cycles.loc[cycle_1, 'nb_avis'] - 1)
haut = profil.loc[profil['indice_moyen'].idxmax()]
bas = profil.loc[profil['indice_moyen'].idxmin()]
hauts_recurrents = profil.loc[profil['indice_min'].gt(100), 'mois'].tolist()
bas_recurrents = profil.loc[profil['indice_max'].lt(100), 'mois'].tolist()
# AJOUT — part des avis du dernier cycle portée par les mois forts récurrents
numeros_forts = profil.index[profil['indice_min'].gt(100)]
part_mois_forts = recent.loc[recent['numero_mois'].isin(numeros_forts), 'nb_avis'].sum() / recent['nb_avis'].sum()


def liste_fr(mots):
    mots = [m.lower() for m in mots]
    return ', '.join(mots[:-1]) + ' et ' + mots[-1] if len(mots) > 1 else (mots[0] if mots else 'aucun')


if round(ecarts.min()) == round(ecarts.max()):
    texte_ecart = f'environ {ecarts.min():.0f} %'
else:
    texte_ecart = f'{ecarts.min():.0f} à {ecarts.max():.0f} %'

conclusion = (
    f'Au {DATE_REFERENCE:%d/%m/%Y}, {nombre_fr(n_actives)} annonces sur {nombre_fr(n_total)} '
    f'({pct_fr(taux_actives)}) ont au moins un avis retenu sur les 365 derniers jours. '
    f'Avec une définition plus stricte, on en compte entre {nombre_fr(variantes["nb_annonces"].min())} '
    f'et {nombre_fr(variantes["nb_annonces"].max())} ({texte_ecart}). '
    'Selon notre définition, environ la moitié des annonces retenues présentent un signe d’activité récente. '
    'Les autres ne peuvent pas être déclarées inactives avec certitude. Ces écarts sont relatifs au nombre '
    'retenu, et non exprimés en points de pourcentage.\n\n'
    f'Sur le cycle complet {cycle_2}, le maximum mensuel est observé en {pic["mois"]} '
    f'avec {nombre_fr(pic["nb_avis"])} avis, et le minimum en {creux["mois"]} '
    f'avec {nombre_fr(creux["nb_avis"])} avis. Le volume total évolue de '
    + f'{variation:+.1f}'.replace('.', ',') + ' % par rapport au cycle précédent, dans la cohorte retenue.\n\n'
    f'Après correction de la longueur des mois et normalisation par cycle, {haut["mois"].lower()} a '
    f'l’indice moyen le plus élevé ({haut["indice_moyen"]:.0f}) et {bas["mois"].lower()} le plus faible '
    f'({bas["indice_moyen"]:.0f}). Les mois au-dessus de 100 dans les deux cycles '
    f'({liste_fr(hauts_recurrents)}) concentrent {pct_fr(100 * part_mois_forts, 0)} des avis du dernier cycle : '
    'Cette part des avis ne mesure pas leur contribution aux revenus. '
    f'Les mois sous 100 dans les deux cycles ({liste_fr(bas_recurrents)}) sont plus calmes ; les plus creux '
    f'({liste_fr(profil.loc[profil["indice_moyen"].lt(90), "mois"].tolist())}, indice moyen inférieur à 90) '
    'peuvent être examinés pour planifier l’entretien, après vérification des réservations réelles '
    'et des contraintes du logement.\n\n'
    'Limites. Ces résultats décrivent les avis de la cohorte conservée, pas les réservations réelles '
    'ni l’ensemble du marché historique. Le revenu dépend aussi des prix, observés à une seule date : '
    'ces données ne permettent donc pas de mesurer les revenus mensuels. '
    'Une éventuelle tendance dans le temps peut influencer le profil mensuel. La hausse entre cycles '
    'ne démontre pas une progression à l’intérieur de chaque cycle. Deux cycles sont insuffisants '
    'pour confirmer une saisonnalité stable. '
    'Les résultats restent provisoires jusqu’à validation du nettoyage.'
)
if mois_incomplet:
    conclusion += f' Le mois {mois_reference} est incomplet et exclu des comparaisons de mois complets.'
print(conclusion)

### 5. Enregistrer les sorties
Les exports contiennent le classement d'activité par annonce, le bilan, la série mensuelle,
le profil saisonnier, la conclusion, quatre graphiques interactifs et deux figures Matplotlib réunis dans un fichier HTML.
Les deux figures Matplotlib sont également exportées en PNG, avec un CSV de sensibilité.
Le HTML est autonome et ne contient que des résultats agrégés ; les identifiants restent
uniquement dans le CSV interne. B1 et B2 ne sont pas modifiés.

Source : données Inside Airbnb de Lyon, licence CC BY 4.0, altérées pour le cours ;
exports provisoires B2. Méthodes : résumé de cours fourni, Pandas et visualisations Plotly / Matplotlib.


In [ ]:
import html
import io
import base64
SORTIES.mkdir(parents=True, exist_ok=True)
annonces[['id', 'nb_avis_365j', 'dernier_avis_retenu', 'active_365j']].to_csv(
    SORTIES / f'activite_annonces_{STATUT}.csv', index=False, encoding='utf-8-sig')
bilan_activite.to_csv(SORTIES / f'bilan_activite_{STATUT}.csv', index=False, encoding='utf-8-sig')
mensuel.drop(columns='periode').to_csv(SORTIES / f'serie_mensuelle_{STATUT}.csv', index=False, encoding='utf-8-sig')
profil.to_csv(SORTIES / f'profil_saisonnier_{STATUT}.csv', encoding='utf-8-sig')
if mois_incomplet:
    partiel.to_csv(SORTIES / f'mois_partiel_{STATUT}.csv', encoding='utf-8-sig')
(SORTIES / f'conclusion_q1_{STATUT}.txt').write_text(conclusion, encoding='utf-8')
figures = [fig_activite, fig_volume, fig_annonces, fig_saison]
blocs = [fig.to_html(full_html=False, include_plotlyjs=(i == 0)) for i, fig in enumerate(figures)]
for figure, titre in [(fig_sensibilite, 'Sensibilité à la définition'),
                      (fig_profil, 'Profil mensuel : moyenne et étendue des deux cycles')]:
    tampon = io.BytesIO()
    figure.savefig(tampon, format='png', dpi=150, bbox_inches='tight')
    encodage = base64.b64encode(tampon.getvalue()).decode('ascii')
    blocs.append(f'<h2>{html.escape(titre)}</h2><img style="max-width:100%;height:auto" '
                 f'alt="{html.escape(titre)}" src="data:image/png;base64,{encodage}">')
blocs.append('<p>Les traits du profil indiquent le minimum et le maximum des deux cycles, '
             'pas des intervalles de confiance.</p>')
page = ('<!doctype html><html lang="fr"><head><meta charset="utf-8">'
        '<meta name="viewport" content="width=device-width,initial-scale=1">'
        '<title>Lyon — Activité et saisonnalité</title><style>'
        'body{font-family:Arial,sans-serif;margin:30px auto;max-width:1100px;padding:0 20px;color:#243444;}'
        'p{line-height:1.6} .note{background:#eef5f6;padding:18px;border-radius:8px;}'
        '</style></head><body><h1>Lyon — Activité et saisonnalité</h1>'
        '<p class="note">Résultats provisoires · Activité définie par les avis sur 365 jours · '
        f'Référence : {DATE_REFERENCE:%d/%m/%Y} · Mois incomplet exclu des comparaisons.</p>'
        + ''.join(blocs) + '<h2>Interprétation</h2><p>'
        + html.escape(conclusion).replace('\n\n', '</p><p>') + '</p></body></html>')
(SORTIES / 'B3_Q1_graphiques_Lyon.html').write_text(page, encoding='utf-8')
print('Sorties enregistrées dans :', SORTIES.resolve())


In [ ]:
# AJOUT — Exports des éléments ajoutés
sensibilite.to_csv(SORTIES / f'sensibilite_definition_{STATUT}.csv', index=False, encoding='utf-8-sig')
fig_sensibilite.savefig(SORTIES / 'q1_sensibilite_definition.png', dpi=150, bbox_inches='tight')
fig_profil.savefig(SORTIES / 'q1_profil_saisonnier.png', dpi=150, bbox_inches='tight')
print('Exports ajoutés : sensibilite_definition, q1_sensibilite_definition.png, q1_profil_saisonnier.png')

## B3 — Q2 : concurrence et concentration

Hôtes multi-annonces, top 10 %, Lorenz et Gini

Objectif : savoir comment les annonces se répartissent entre les hôtes. On mesure la
concentration du **nombre d’annonces** et celle du **revenu estimé** (section 6 bis, **[Ajout]**), 

**Entrée complémentaire pour les revenus :** `estimated_revenue_l365d`, relue dans la table nettoyée `listings_propre`.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

DOSSIER_B2 = SORTIES_B / 'sorties_b2_jointure'
SORTIES = SORTIES_B / 'sorties_b3_q2'
STATUT = 'validee'
DATE_REFERENCE = pd.Timestamp('2026-09-19')  # Même référence que B2 et Q1.
fichier = DOSSIER_B2 / f'listings_avec_avis_{STATUT}.csv'
if not fichier.is_file():
    raise FileNotFoundError(f'Fichier absent : {fichier.resolve()} ; exécuter B2 ou adapter DOSSIER_B2.')
annonces = pd.read_csv(fichier, dtype={'id': 'string', 'host_id': 'string'})
assert len(annonces) > 0
assert annonces['id'].notna().all() and annonces['id'].is_unique, 'B2 doit fournir une ligne par annonce.'
annonces['nb_avis_365j'] = pd.to_numeric(annonces['nb_avis_365j'], errors='raise')
assert annonces['nb_avis_365j'].notna().all()
assert annonces['nb_avis_365j'].ge(0).all() and annonces['nb_avis_365j'].mod(1).eq(0).all()
dates_collecte = pd.to_datetime(annonces['last_scraped'], errors='raise')
assert dates_collecte.notna().all() and dates_collecte.eq(DATE_REFERENCE).all(), 'Vérifier la référence de B2.'

def nombre_fr(x):
    return f'{x:,.0f}'.replace(',', ' ')

def decimal_fr(x, decimales=1):
    return f'{x:.{decimales}f}'.replace('.', ',')

print('Annonces chargées :', nombre_fr(len(annonces)))

### 1. Définir le périmètre et contrôler les identifiants
Un **hôte** est identifié par `host_id`, jamais par son nom. Un hôte **multi-annonces**
possède au moins deux identifiants d’annonce distincts dans le fichier lyonnais retenu.
Il peut être propriétaire, gestionnaire ou conciergerie : plusieurs annonces ne prouvent pas un statut professionnel.

Analyse principale : toutes les annonces retenues par B2, avec ou sans avis récent et avec ou sans prix exploitable.
Les identifiants d’hôte manquants ou de format invalide sont isolés, pas regroupés sous un « hôte inconnu ».
Les contrôles de format ne garantissent pas que l’identité est correcte ; cela dépend du nettoyage A5.

In [ ]:
annonces['host_id_source'] = annonces['host_id']
annonces['host_id'] = annonces['host_id'].str.strip()
hote_valide = annonces['host_id'].str.fullmatch(r'[1-9][0-9]*').fillna(False)
rejets_hotes = annonces.loc[~hote_valide, ['id', 'host_id_source']].copy()
rejets_hotes['motif'] = 'host_id_manquant_ou_format_invalide'
base = annonces.loc[hote_valide].copy()
assert len(base) > 0, 'Aucun hôte exploitable.'
assert len(base) + len(rejets_hotes) == len(annonces)
print('Annonces avec hôte exploitable :', nombre_fr(len(base)))
print('Annonces isolées :', nombre_fr(len(rejets_hotes)))
print('Couverture :', decimal_fr(100 * len(base) / len(annonces)), '%')

### 2. Compter les annonces par hôte
`groupby('host_id')['id'].nunique()` crée une ligne par hôte et compte ses annonces distinctes.
On calcule séparément :

- la **part des hôtes** ayant plusieurs annonces ;
- la **part des annonces** rattachées à ces hôtes.

Les dénominateurs sont différents : ne pas confondre ces deux pourcentages.

In [ ]:
hotes = base.groupby('host_id', as_index=False).agg(nb_annonces=('id', 'nunique'))
hotes['multi_annonce'] = hotes['nb_annonces'].ge(2)
hotes['groupe'] = np.where(hotes['multi_annonce'], 'Multi-annonces (2 ou plus)', 'Une seule annonce')
n_hotes = len(hotes)
n_annonces = int(hotes['nb_annonces'].sum())
assert hotes['host_id'].is_unique and hotes['nb_annonces'].ge(1).all()
assert n_annonces == len(base)
bilan = hotes.groupby('groupe').agg(nb_hotes=('host_id', 'size'), nb_annonces=('nb_annonces', 'sum'))
bilan = bilan.reindex(['Une seule annonce', 'Multi-annonces (2 ou plus)'], fill_value=0)
bilan['part_hotes_pct'] = 100 * bilan['nb_hotes'] / n_hotes
bilan['part_annonces_pct'] = 100 * bilan['nb_annonces'] / n_annonces
assert bilan['nb_hotes'].sum() == n_hotes and bilan['nb_annonces'].sum() == n_annonces
print(bilan.round(2).to_string())

comparaison = bilan.reset_index().melt(
    id_vars='groupe', value_vars=['part_hotes_pct', 'part_annonces_pct'],
    var_name='population', value_name='part_pct')
comparaison['population'] = comparaison['population'].map({
    'part_hotes_pct': 'Part des hôtes', 'part_annonces_pct': 'Part des annonces'})
fig_multi = px.bar(comparaison, x='groupe', y='part_pct', color='population', barmode='group',
    text_auto='.1f', template='plotly_white',
    title='Hôtes multi-annonces : comparer leur poids parmi les hôtes et parmi les annonces',
    labels={'groupe': '', 'part_pct': 'Part (%)', 'population': 'Dénominateur'})
fig_multi.update_yaxes(range=[0, 100])
fig_multi.show()

### 3. Décrire la taille des portefeuilles locaux
Les classes 1, 2, 3–5, 6–10 et 11 annonces ou plus rendent la distribution lisible.
Le graphique compte les hôtes ; le tableau donne aussi le nombre d’annonces correspondant.
Le terme « portefeuille » désigne les annonces associées au compte dans ce périmètre, pas les biens possédés.

In [ ]:
classes = ['1', '2', '3–5', '6–10', '11 ou plus']
hotes['classe'] = pd.cut(hotes['nb_annonces'], bins=[0, 1, 2, 5, 10, np.inf], labels=classes)
distribution = hotes.groupby('classe', observed=False).agg(
    nb_hotes=('host_id', 'size'), nb_annonces=('nb_annonces', 'sum')).reset_index()
distribution['part_hotes_pct'] = 100 * distribution['nb_hotes'] / n_hotes
assert distribution['nb_hotes'].sum() == n_hotes
print(distribution.round(2).to_string(index=False))
print('Médiane :', hotes['nb_annonces'].median(), '; maximum :', hotes['nb_annonces'].max())
fig_distribution = px.bar(distribution, x='classe', y='nb_hotes', text='nb_hotes',
    template='plotly_white', category_orders={'classe': classes},
    title='Nombre d’hôtes selon leur nombre d’annonces à Lyon',
    labels={'classe': 'Annonces par hôte', 'nb_hotes': 'Nombre d’hôtes'},
    hover_data={'nb_annonces': True, 'part_hotes_pct': ':.1f'})
fig_distribution.update_traces(marker_color='#087F8C')
fig_distribution.show()

### 4. Mesurer le poids du top 10 % des hôtes
Il s’agit des **10 % d’hôtes ayant le plus d’annonces**, pas des 10 % d’annonces les plus chères.
On trie par nombre d’annonces décroissant et on retient `k = ceil(0.10 × nombre d’hôtes)`.
L’arrondi supérieur permet un nombre entier d’hôtes ; la part exacte peut dépasser très légèrement 10 %.

En cas d’ex æquo au seuil, on utilise `host_id` comme simple règle de tri reproductible.
Le choix entre ces ex æquo ne change pas le nombre total d’annonces du top, puisque leurs comptes sont égaux.
On montre aussi une variante qui inclut **tous** les ex æquo : elle ne représente plus nécessairement 10 % des hôtes.

In [ ]:
classement = hotes.sort_values(['nb_annonces', 'host_id'], ascending=[False, True]).reset_index(drop=True)
k = math.ceil(0.10 * n_hotes)
classement['rang'] = np.arange(1, n_hotes + 1)
classement['top_10_pct'] = classement['rang'].le(k)
seuil = int(classement.iloc[k - 1]['nb_annonces'])
n_superieurs = int(classement['nb_annonces'].gt(seuil).sum())
n_exaequo = int(classement['nb_annonces'].eq(seuil).sum())
n_exaequo_retenus = k - n_superieurs
top_annonces = int(classement.loc[classement['top_10_pct'], 'nb_annonces'].sum())
part_top = 100 * top_annonces / n_annonces
avec_exaequo = classement['nb_annonces'].ge(seuil)
comparaison_top = pd.DataFrame([
    {'regle': 'Top 10 % (arrondi supérieur)', 'nb_hotes': k, 'nb_annonces': top_annonces},
    {'regle': 'Avec tous les ex æquo au seuil', 'nb_hotes': int(avec_exaequo.sum()),
     'nb_annonces': int(classement.loc[avec_exaequo, 'nb_annonces'].sum())}
])
comparaison_top['part_hotes_pct'] = 100 * comparaison_top['nb_hotes'] / n_hotes
comparaison_top['part_annonces_pct'] = 100 * comparaison_top['nb_annonces'] / n_annonces
assert classement['top_10_pct'].sum() == k and top_annonces <= n_annonces
print(comparaison_top.round(2).to_string(index=False))
print(f'Seuil : {seuil} annonces ; {n_superieurs} hôtes strictement au-dessus ; '
      f'{n_exaequo_retenus} retenus parmi {n_exaequo} ex æquo au seuil.')

parts_top = pd.DataFrame({'groupe': ['Top 10 % des hôtes', 'Autres hôtes'],
    'part_annonces_pct': [part_top, 100 - part_top],
    'nb_annonces': [top_annonces, n_annonces - top_annonces]})
fig_top = px.bar(parts_top, x='groupe', y='part_annonces_pct', text_auto='.1f',
    template='plotly_white', hover_data=['nb_annonces'],
    title=f'Les {nombre_fr(k)} hôtes du top 10 % regroupent {decimal_fr(part_top)} % des annonces',
    labels={'groupe': '', 'part_annonces_pct': 'Part des annonces (%)'})
fig_top.update_yaxes(range=[0, 100])
fig_top.update_traces(marker_color='#C16E32')
fig_top.show()

### 5. Courbe de Lorenz et indice de Gini
Pour Lorenz, on trie cette fois les hôtes du **plus petit au plus grand** nombre d’annonces.
L’axe horizontal représente la part cumulée des hôtes, l’axe vertical leur part cumulée des annonces.
La diagonale représente l’égalité : chaque hôte aurait le même nombre d’annonces.
Plus la courbe s’écarte en dessous de cette diagonale, plus la répartition est concentrée.

Pour les nombres d’annonces triés $x_{(1)} \leq \cdots \leq x_{(n)}$, on calcule :

\[
G = \frac{2\sum_{i=1}^{n} i x_{(i)}}{n\sum_{i=1}^{n}x_{(i)}} - \frac{n+1}{n}
\]

**Gini = 0** signifie une égalité parfaite ; plus il se rapproche de 1, plus la concentration est forte.
On utilise la formule descriptive standard, sans correction `n/(n-1)`.
La population comprend uniquement les hôtes ayant au moins une annonce retenue, pas les personnes sans annonce.
Le Gini porte sur des nombres d’annonces, et ne mesure pas une inégalité de revenus.

In [ ]:
def calcul_gini(valeurs):
    x = np.sort(np.asarray(valeurs, dtype=float))
    if x.ndim != 1 or len(x) == 0 or not np.isfinite(x).all() or (x < 0).any() or x.sum() <= 0:
        raise ValueError('Le Gini nécessite des valeurs finies, non négatives et de somme positive.')
    n = len(x)
    rangs = np.arange(1, n + 1)
    return float(2 * (rangs * x).sum() / (n * x.sum()) - (n + 1) / n)

# Petits exemples dont le résultat est connu, puis contrôle par l’aire de Lorenz.
assert np.isclose(calcul_gini([2, 2, 2]), 0)
assert np.isclose(calcul_gini([1, 3]), 0.25)
x = np.sort(hotes['nb_annonces'].to_numpy())
lorenz = pd.DataFrame({
    'part_hotes': np.arange(n_hotes + 1) / n_hotes,
    'part_annonces': np.r_[0, np.cumsum(x) / x.sum()]
})
gini = calcul_gini(x)
aire = np.sum(np.diff(lorenz['part_hotes']) *
              (lorenz['part_annonces'].to_numpy()[1:] + lorenz['part_annonces'].to_numpy()[:-1]) / 2)
assert np.isclose(gini, 1 - 2 * aire)
assert 0 <= gini < 1
assert np.all(np.diff(lorenz['part_annonces']) >= 0)
assert np.all(lorenz['part_annonces'] <= lorenz['part_hotes'] + 1e-12)
assert np.allclose(lorenz.iloc[0], [0, 0]) and np.allclose(lorenz.iloc[-1], [1, 1])
point_hors_top = lorenz.iloc[n_hotes - k]
assert np.isclose(100 * (1 - point_hors_top['part_annonces']), part_top)
print('Indice de Gini :', decimal_fr(gini, 4))

fig_lorenz = go.Figure()
fig_lorenz.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode='lines', name='Égalité parfaite',
    line=dict(color='#888888', dash='dash')))
fig_lorenz.add_trace(go.Scatter(x=100 * lorenz['part_hotes'], y=100 * lorenz['part_annonces'],
    mode='lines', name='Répartition observée', line=dict(color='#087F8C', width=3),
    hovertemplate='%{x:.2f} % des hôtes : %{y:.2f} % des annonces<extra></extra>'))
fig_lorenz.add_trace(go.Scatter(x=[100 * point_hors_top['part_hotes']],
    y=[100 * point_hors_top['part_annonces']], mode='markers', name='Limite du top 10 %',
    marker=dict(size=10, color='#C16E32')))
fig_lorenz.update_layout(template='plotly_white', title=f'Concentration des annonces — Gini : {decimal_fr(gini, 3)}',
    xaxis_title='Part cumulée des hôtes, du plus petit au plus grand portefeuille (%)',
    yaxis_title='Part cumulée des annonces (%)', height=600)
fig_lorenz.update_xaxes(range=[0, 100])
fig_lorenz.update_yaxes(range=[0, 100], scaleanchor='x', scaleratio=1)
fig_lorenz.show()

### 6. Variante : uniquement les annonces actives selon Q1
On refait les calculs sur les annonces ayant au moins un avis retenu sur 365 jours.
On **recompte** les annonces de chaque hôte dans ce sous-ensemble : un hôte multi-annonces au total
peut n’avoir qu’une seule annonce active selon les avis. Les hôtes sans annonce active sont absents de la variante.
Les deux lignes n’ont donc pas la même population ; cette comparaison décrit l’effet du périmètre,
pas une évolution dans le temps. La définition principale reste celle de toutes les annonces retenues.

In [ ]:
def resumer_perimetre(table, nom):
    effectifs = table.groupby('host_id')['id'].nunique().sort_values(ascending=False)
    n = len(effectifs)
    if n == 0:
        return {'perimetre': nom, 'nb_hotes': 0, 'nb_annonces': 0,
                'part_hotes_multi_pct': np.nan, 'part_annonces_multi_pct': np.nan,
                'nb_hotes_top': 0, 'part_annonces_top_pct': np.nan, 'gini': np.nan}
    k_local = math.ceil(0.1 * n)
    return {'perimetre': nom, 'nb_hotes': n, 'nb_annonces': int(effectifs.sum()),
        'part_hotes_multi_pct': 100 * effectifs.ge(2).mean(),
        'part_annonces_multi_pct': 100 * effectifs[effectifs.ge(2)].sum() / effectifs.sum(),
        'nb_hotes_top': k_local,
        'part_annonces_top_pct': 100 * effectifs.iloc[:k_local].sum() / effectifs.sum(),
        'gini': calcul_gini(effectifs)}

base_active = base.loc[base['nb_avis_365j'].gt(0)]
sensibilite = pd.DataFrame([
    resumer_perimetre(base, 'Toutes les annonces retenues avec hôte exploitable'),
    resumer_perimetre(base_active, 'Annonces actives selon les avis sur 365 jours')
])
assert np.isclose(sensibilite.iloc[0]['gini'], gini)
assert np.isclose(sensibilite.iloc[0]['part_annonces_top_pct'], part_top)
print(sensibilite.round(3).to_string(index=False))

### 6 bis. Concentration du revenu estimé

Cette section complète l’analyse des nombres d’annonces par celle des revenus estimés.

- **Source :** `estimated_revenue_l365d`, relue dans la table nettoyée `listings_propre` ; il s’agit d’une estimation fournie, pas d’un revenu réel constaté ni d’un revenu net. Elle n’est pas recalculée à partir des avis nettoyés de B2 ou des variables de B1.
- **Périmètre :** annonces ayant au moins un avis retenu sur 365 jours selon Q1. Une annonce sans avis récent retenu peut avoir un revenu estimé positif dans la source ; les deux indicateurs peuvent donc diverger. L’absence d’avis ne prouve pas une absence de réservation.
- **Valeurs manquantes :** `sum(min_count=1)` préserve une valeur manquante lorsque toutes les estimations d’un hôte sont absentes. Ces hôtes sont exclus du calcul du revenu. Lorsqu’une partie seulement des annonces a une estimation, la somme conservée est **partielle** : elle ne représente pas le revenu total de l’hôte. Les effectifs de ces trois situations sont contrôlés ci-dessous.
- **Dénominateurs :** la part des annonces porte sur toutes les annonces actives ; la part du revenu porte seulement sur les montants renseignés. Ces parts ne décrivent donc pas exactement la même couverture. Le statut multi-annonces reste défini sur l’ensemble du portefeuille retenu en section 2.
- **Top 10 % par revenu :** classement des hôtes ayant au moins un revenu estimé renseigné, selon leur somme disponible, avec arrondi supérieur. Ce groupe diffère du top 10 % par nombre d’annonces.
- **Limites :** les valeurs absentes et les sommes partielles peuvent modifier le classement, les parts et le Gini. Leur effet global n’est pas connu ; remplacer les valeurs absentes par zéro n’est pas une correction. Le Gini obtenu avec cette convention est seulement une comparaison méthodologique.


In [ ]:
# AJOUT — Revenu estimé par hôte : la consigne de Q2 porte aussi sur le revenu (annexe D).
import matplotlib.pyplot as plt


revenus_source = listings_propre[['id', 'estimated_revenue_l365d']].copy()   # table nettoyée (A6)
revenus_source['id'] = revenus_source['id'].astype('string')
revenus_source['revenu_estime'] = pd.to_numeric(revenus_source['estimated_revenue_l365d'], errors='coerce')
# Les identifiants répétés du fichier brut ont tous le même revenu : on le vérifie avant de dédoublonner.
assert revenus_source.groupby('id')['revenu_estime'].nunique(dropna=False).le(1).all()
revenus_source = revenus_source.drop_duplicates('id')[['id', 'revenu_estime']]

# Périmètre de l'annexe D : les annonces actives (au moins un avis sur 365 jours, définition de Q1).
actives_q2 = base.loc[base['nb_avis_365j'].gt(0)].merge(revenus_source, on='id', how='left', validate='one_to_one', indicator=True)
assert actives_q2['_merge'].eq('both').all(), 'Certaines annonces actives sont absentes du fichier source.'
actives_q2 = actives_q2.drop(columns='_merge')
assert len(actives_q2) == int(base['nb_avis_365j'].gt(0).sum())
# Hôte multi-annonces : même définition qu'en section 2 (portefeuille dans le périmètre principal).
actives_q2['multi_annonce'] = actives_q2['host_id'].map(hotes.set_index('host_id')['multi_annonce'])
assert actives_q2['multi_annonce'].notna().all()
print('Annonces actives :', nombre_fr(len(actives_q2)),
      '| revenu estimé manquant :', nombre_fr(actives_q2['revenu_estime'].isna().sum()))

# Revenu par hôte. min_count=1 : un hôte dont aucune annonce n'a de revenu connu reste MANQUANT.
# Avec un simple .sum(), il serait compté à 0 € et gonflerait artificiellement la concentration.
revenu_hote = actives_q2.groupby('host_id').agg(
    revenu=('revenu_estime', lambda s: s.sum(min_count=1)),
    multi_annonce=('multi_annonce', 'first'),
    nb_annonces_actives=('id', 'nunique'),
    nb_revenus_connus=('revenu_estime', 'count'),
)
revenu_hote['couverture_revenu'] = np.select(
    [revenu_hote['nb_revenus_connus'].eq(0),
     revenu_hote['nb_revenus_connus'].eq(revenu_hote['nb_annonces_actives'])],
    ['Aucun revenu renseigné', 'Toutes les annonces actives renseignées'],
    default='Somme partielle')
audit_revenu_hotes = revenu_hote.copy()
n_partiels = int(revenu_hote['couverture_revenu'].eq('Somme partielle').sum())
n_complets = int(revenu_hote['couverture_revenu'].eq('Toutes les annonces actives renseignées').sum())
n_sans_revenu = int(revenu_hote['revenu'].isna().sum())
assert n_partiels + n_complets + n_sans_revenu == len(revenu_hote)
controle_source = base[['id', 'nb_avis_365j']].merge(revenus_source, on='id', how='left', validate='one_to_one')
n_sans_avis_revenu_positif = int((controle_source['nb_avis_365j'].eq(0) & controle_source['revenu_estime'].gt(0)).sum())
print('Hôtes avec somme partielle :', n_partiels)
print('Annonces sans avis récent retenu mais avec revenu source positif :', n_sans_avis_revenu_positif)
revenu_hote = revenu_hote.dropna(subset=['revenu'])
assert revenu_hote['revenu'].gt(0).all()

# 1. Part du revenu estimé détenue par les hôtes multi-annonces
part_revenu_multi = revenu_hote.loc[revenu_hote['multi_annonce'], 'revenu'].sum() / revenu_hote['revenu'].sum()
part_annonces_multi_actives = actives_q2['multi_annonce'].mean()

# 2. Top 10 % des hôtes classés par revenu estimé (peu d'ex æquo, contrairement au nombre d'annonces)
k_revenu = math.ceil(0.10 * len(revenu_hote))
top_revenu = revenu_hote.reset_index().sort_values(['revenu', 'host_id'], ascending=[False, True]).head(k_revenu)
part_top_revenu = top_revenu['revenu'].sum() / revenu_hote['revenu'].sum()

# 3. Lorenz et Gini du revenu par hôte : ta formule, contrôlée par la méthode des trapèzes de l'annexe D
r = np.sort(revenu_hote['revenu'].to_numpy(dtype=float))
lorenz_revenu = pd.DataFrame({'part_hotes': np.arange(len(r) + 1) / len(r),
                              'part_revenu': np.r_[0, np.cumsum(r) / r.sum()]})
gini_revenu = calcul_gini(r)
aire_revenu = np.sum(np.diff(lorenz_revenu['part_hotes'])
                     * (lorenz_revenu['part_revenu'].to_numpy()[1:] + lorenz_revenu['part_revenu'].to_numpy()[:-1]) / 2)
assert np.isclose(gini_revenu, 1 - 2 * aire_revenu)
gini_naif = calcul_gini(actives_q2.groupby('host_id')['revenu_estime'].sum())

revenu_bilan = pd.DataFrame([
    ('Hôtes actifs avec au moins une estimation renseignée', nombre_fr(len(revenu_hote))),
    ('Dont hôtes avec somme partielle', nombre_fr(n_partiels)),
    ('Dont hôtes avec toutes les annonces actives renseignées', nombre_fr(n_complets)),
    ('Hôtes actifs sans aucun revenu connu (exclus)', nombre_fr(n_sans_revenu)),
    ('Part des annonces actives détenue par les multi-annonces', decimal_fr(100 * part_annonces_multi_actives) + ' %'),
    ('Part du revenu estimé détenue par les multi-annonces', decimal_fr(100 * part_revenu_multi) + ' %'),
    (f'Part du revenu détenue par le top 10 % ({nombre_fr(k_revenu)} hôtes)', decimal_fr(100 * part_top_revenu) + ' %'),
    ("Seuil d'entrée dans le top 10 %", nombre_fr(top_revenu['revenu'].min()) + ' €'),
    ('Gini du revenu par hôte', decimal_fr(gini_revenu, 2)),
    ('Gini si les revenus manquants comptaient 0 € (erreur à éviter)', decimal_fr(gini_naif, 2)),
], columns=['indicateur', 'valeur'])
print(revenu_bilan.to_string(index=False))

# Graphique Matplotlib (compte pour B4) : titre = constat
part_90 = np.interp(0.9, lorenz_revenu['part_hotes'], lorenz_revenu['part_revenu'])
fig_lorenz_revenu, ax = plt.subplots(figsize=(6.2, 5.6))
ax.fill_between(lorenz_revenu['part_hotes'], lorenz_revenu['part_revenu'], lorenz_revenu['part_hotes'],
                color='#087F8C', alpha=0.12)
ax.plot(lorenz_revenu['part_hotes'], lorenz_revenu['part_revenu'], color='#087F8C', linewidth=2,
        label='Répartition observée')
ax.plot([0, 1], [0, 1], linestyle='--', color='grey', label='Égalité parfaite')
ax.scatter([0.9], [part_90], color='#C16E32', zorder=3)
ax.annotate(f'90 % des hôtes\n→ {decimal_fr(100 * part_90, 0)} % du revenu', xy=(0.9, part_90),
            xytext=(0.2, 0.56), fontsize=9, arrowprops={'arrowstyle': '->', 'color': '#C16E32'})
ax.set_title(f"Le top 10 % des hôtes par somme disponible concentre {decimal_fr(100 * part_top_revenu, 0)} %\n"
             f"du revenu estimé renseigné (Gini = {decimal_fr(gini_revenu, 2)})", loc='left', fontsize=11)
ax.set_xlabel('Part cumulée des hôtes (du plus petit au plus grand revenu)')
ax.set_ylabel('Part cumulée du revenu estimé')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.spines[['top', 'right']].set_visible(False)
ax.legend(loc='upper left', frameon=False)
plt.tight_layout()
plt.show()

SORTIES.mkdir(parents=True, exist_ok=True)
audit_revenu_hotes.to_csv(SORTIES / f'audit_couverture_revenu_hotes_{STATUT}.csv', encoding='utf-8-sig')
revenu_bilan.to_csv(SORTIES / f'revenu_concentration_{STATUT}.csv', index=False, encoding='utf-8-sig')
lorenz_revenu.to_csv(SORTIES / f'courbe_lorenz_revenu_{STATUT}.csv', index=False, encoding='utf-8-sig')
fig_lorenz_revenu.savefig(SORTIES / 'q2_lorenz_revenu.png', dpi=150, bbox_inches='tight')

### 7. Conclusion et limites
La conclusion est calculée automatiquement. On évite les seuils arbitraires tels que
« Gini inférieur à 0,3 = faible » : l’interprétation doit tenir compte de la population étudiée.

Limites : photographie du fichier lyonnais, données  altérées, comptes distincts pouvant relever d’un même gestionnaire, compte unique pouvant gérer des biens de plusieurs propriétaires.
Le nombre d’annonces ne renseigne directement ni le nombre de logements physiques distincts,
ni le portefeuille mondial de l’hôte, ni ses revenus réels, ni sa conformité réglementaire.

La comparaison entre le Gini des annonces et celui des revenus renseignés porte sur des populations différentes.
Elle ne permet pas d’attribuer les écarts à la performance des annonces. Les revenus sont des estimations
issues du fichier source et peuvent être incomplets ; ils ne doivent pas être présentés comme des gains garantis.

In [ ]:
multi = bilan.loc['Multi-annonces (2 ou plus)']
var_active = sensibilite.iloc[1]
conclusion = (
    f'Dans le périmètre retenu, {nombre_fr(n_hotes)} hôtes regroupent {nombre_fr(n_annonces)} annonces. '
    f'{nombre_fr(multi["nb_hotes"])} hôtes ont plusieurs annonces, soit '
    f'{decimal_fr(multi["part_hotes_pct"])} % des hôtes. Ils regroupent '
    f'{nombre_fr(multi["nb_annonces"])} annonces, soit {decimal_fr(multi["part_annonces_pct"])} % de l’offre étudiée.\n\n'
    f'Les {nombre_fr(k)} hôtes du top 10 % ({decimal_fr(100*k/n_hotes, 2)} % des hôtes après arrondi) '
    f'regroupent {nombre_fr(top_annonces)} annonces, soit {decimal_fr(part_top)} % des annonces. '
    f'Le seuil est de {seuil} annonces ; {nombre_fr(n_exaequo_retenus)} hôtes parmi '
    f'{nombre_fr(n_exaequo)} ex æquo sont retenus au seuil. Inclure tous les ex æquo porterait le groupe à '
    f'{decimal_fr(comparaison_top.iloc[1]["part_hotes_pct"])} % des hôtes, pour '
    f'{decimal_fr(comparaison_top.iloc[1]["part_annonces_pct"])} % des annonces : ce ne serait plus le même top 10 %.\n\n'
    f'La courbe de Lorenz se situe sous la diagonale d’égalité et le Gini vaut {decimal_fr(gini, 3)}. '
    'Les annonces sont donc inégalement réparties entre les hôtes. Le top 10 % et la part des multi-annonces '
    'précisent cette concentration sans appliquer une étiquette arbitraire au Gini.\n\n'
)
if var_active['nb_hotes'] > 0:
    conclusion += (f'En limitant le calcul aux {nombre_fr(var_active["nb_annonces"])} annonces actives selon les avis, '
        f'on retient {nombre_fr(var_active["nb_hotes"])} hôtes ; le top 10 % regroupe '
        f'{decimal_fr(var_active["part_annonces_top_pct"])} % de ces annonces et le Gini vaut '
        f'{decimal_fr(var_active["gini"], 3)}. La population et les portefeuilles ont été recalculés.\n\n')
conclusion += (f'{nombre_fr(len(rejets_hotes))} annonce(s) ont été isolées pour identifiant d’hôte inexploitable. '
    'Les sections 1 à 6 décrivent des comptes d’annonces, pas la propriété. '
    'Ces résultats restent provisoires jusqu’à validation du nettoyage par le membre 1.')
# Interprétation du revenu disponible et de sa couverture.
conclusion += ('\n\nRevenu estimé renseigné : '
    f'{nombre_fr(len(revenu_hote))} hôtes actifs ont au moins une estimation disponible. '
    f'Parmi eux, {nombre_fr(n_complets)} ont toutes leurs annonces actives renseignées et '
    f'{nombre_fr(n_partiels)} ont une somme partielle. '
    f'Les {nombre_fr(n_sans_revenu)} hôtes sans aucune estimation sont exclus.\n\n'
    f'Les hôtes multi-annonces regroupent {decimal_fr(100 * part_annonces_multi_actives)} % de toutes les annonces actives '
    f'et {decimal_fr(100 * part_revenu_multi)} % des revenus estimés renseignés. '
    'Les dénominateurs ont des couvertures différentes. '
    f'Les {nombre_fr(k_revenu)} hôtes du top 10 % par somme disponible concentrent '
    f'{decimal_fr(100 * part_top_revenu)} % des revenus renseignés ; le seuil retenu est '
    f'{nombre_fr(top_revenu["revenu"].min())} € estimés sur les 365 derniers jours.\n\n'
    f'Le Gini de ces sommes vaut {decimal_fr(gini_revenu, 2)}, contre '
    f'{decimal_fr(var_active["gini"], 2)} pour le nombre d’annonces actives. '
    'Le revenu estimé renseigné apparaît plus concentré, mais les populations diffèrent et '
    'cette comparaison ne permet pas d’identifier les causes de l’écart. '
    'Les estimations absentes et les sommes partielles peuvent modifier les classements et les indices, '
    'sans que l’on connaisse le sens global du biais. '
    f'La convention consistant à remplacer les revenus absents par zéro donnerait ici un Gini de '
    f'{decimal_fr(gini_naif, 2)} ; ce n’est pas une correction justifiée.\n\n'
    f'Dans la source, {nombre_fr(n_sans_avis_revenu_positif)} annonces sans avis récent retenu ont un revenu estimé positif. '
    'La définition d’activité selon les avis et l’estimation de la source ne sont donc pas interchangeables. '
    'Les montants étudiés sont des estimations disponibles, pas des revenus réels ou nets. '
    'Les résultats restent provisoires jusqu’à validation du nettoyage.')
print(conclusion)


In [ ]:
import html
import io
import base64
SORTIES.mkdir(parents=True, exist_ok=True)
tables = {
    'hotes_classement': classement,
    'bilan_multi_annonces': bilan.reset_index(),
    'distribution_portefeuilles': distribution,
    'comparaison_top_10': comparaison_top,
    'courbe_lorenz': lorenz,
    'sensibilite_perimetre': sensibilite,
    'rejets_host_id': rejets_hotes
}
for nom, table in tables.items():
    table.to_csv(SORTIES / f'{nom}_{STATUT}.csv', index=False, encoding='utf-8-sig')
(SORTIES / f'conclusion_q2_{STATUT}.txt').write_text(conclusion, encoding='utf-8')
figures = [fig_multi, fig_distribution, fig_top, fig_lorenz]
blocs = [fig.to_html(full_html=False, include_plotlyjs=(i == 0)) for i, fig in enumerate(figures)]
tampon = io.BytesIO()
fig_lorenz_revenu.savefig(tampon, format='png', dpi=150, bbox_inches='tight')
image_revenu = base64.b64encode(tampon.getvalue()).decode('ascii')
blocs.append('<h2>Concentration des revenus estimés renseignés</h2>'
             '<img style="max-width:100%;height:auto" alt="Courbe de Lorenz du revenu estimé renseigné" '
             f'src="data:image/png;base64,{image_revenu}">')
page = ('<!doctype html><html lang="fr"><head><meta charset="utf-8">'
        '<meta name="viewport" content="width=device-width,initial-scale=1">'
        '<title>Lyon — Concentration des annonces</title><style>'
        'body{font-family:Arial,sans-serif;max-width:1100px;margin:30px auto;padding:0 20px;color:#243444;}'
        'p{line-height:1.6}.note{padding:18px;background:#eef5f6;border-radius:8px;}'
        '</style></head><body><h1>B3 — Q2 : concentration des annonces à Lyon</h1>'
        '<p class="note">Résultats provisoires · Nombre d’annonces par hôte dans le périmètre B2 · '
        'Deux classements distincts : nombre d’annonces et somme des revenus estimés renseignés.</p>'
        + ''.join(blocs) + '<h2>Conclusion et limites</h2><p>'
        + html.escape(conclusion).replace('\n\n', '</p><p>')
        + '</p><p>Source : Inside Airbnb, CC BY 4.0, données pédagogiques altérées. '
        'Snapshot annoncé le 17/09/2026 ; collecte observée le 19/09/2026.</p></body></html>')
(SORTIES / 'B3_Q2_graphiques_Lyon.html').write_text(page, encoding='utf-8')
print('Résultats enregistrés dans :', SORTIES.resolve())

## B3 — Q3 : où investir

Question de l'investisseur : dans quels arrondissements le revenu, l'occupation et la concurrence sont-ils les plus favorables ?
**Périmètre :** annonces actives selon la définition de Q1 (au moins un avis sur les 365 jours précédant la collecte). Le prix, l'occupation et le revenu viennent de la table nettoyée ; le revenu et le prix médians ne portent que sur les annonces avec un prix publié.

In [ ]:
import plotly.express as px

q3 = listings_propre[["id", "neighbourhood_cleansed", "room_type", "price",
                      "estimated_occupancy_l365d", "estimated_revenue_l365d"]].copy()
activite = listings_avec_avis[["id", "nb_avis_365j"]].copy()
activite["id"] = activite["id"].astype("int64")
q3 = q3.merge(activite, on="id", how="left", validate="one_to_one")
q3 = q3[q3["nb_avis_365j"].fillna(0).gt(0)].copy()
print("Annonces actives :", len(q3), "| avec un prix publié :", int(q3["price"].notna().sum()))

ordre = list(population["arrondissement"])   # 1er, 2e, ..., 9e
par_arrondissement = q3.groupby("neighbourhood_cleansed").agg(
    annonces_actives=("id", "count"),
    annonces_avec_prix=("price", "count"),
    prix_median=("price", "median"),
    occupation_mediane=("estimated_occupancy_l365d", "median"),
    revenu_median=("estimated_revenue_l365d", "median"),
).reindex(ordre)
par_arrondissement = par_arrondissement.join(population.set_index("arrondissement")["population_municipale"])
par_arrondissement["annonces_pour_1000_hab"] = (
    par_arrondissement["annonces_actives"] / par_arrondissement["population_municipale"] * 1000)
par_arrondissement.round(1)

### Détail par type de logement (`pivot_table` avec `room_type`)

In [ ]:
pivot_annonces = pd.pivot_table(q3, index="neighbourhood_cleansed", columns="room_type",
                                values="id", aggfunc="count", fill_value=0).reindex(ordre)
pivot_revenu = pd.pivot_table(q3, index="neighbourhood_cleansed", columns="room_type",
                              values="estimated_revenue_l365d", aggfunc="median").reindex(ordre)
print("Nombre d'annonces actives")
display(pivot_annonces)
print("Revenu médian estimé (€ par an) ; vide si aucune annonce avec prix")
display(pivot_revenu.round(0))

### Classement des arrondissements
Deux critères, de poids égal : **revenu médian élevé** (plus c'est haut, mieux c'est) et **concurrence faible** (moins d'annonces actives pour 1 000 habitants). Le score est la moyenne des deux rangs (1 = le meilleur). On écarte du classement les arrondissements ayant moins de 30 annonces avec prix, dont la médiane est trop instable.

In [ ]:
MIN_ANNONCES_PRIX = 30
classement = par_arrondissement[par_arrondissement["annonces_avec_prix"] >= MIN_ANNONCES_PRIX].copy()
classement["rang_revenu"] = classement["revenu_median"].rank(ascending=False, method="min")
classement["rang_concurrence"] = classement["annonces_pour_1000_hab"].rank(ascending=True, method="min")
classement["score"] = (classement["rang_revenu"] + classement["rang_concurrence"]) / 2
classement = classement.sort_values(["score", "revenu_median"], ascending=[True, False])
display(classement[["annonces_actives", "revenu_median", "occupation_mediane", "annonces_pour_1000_hab",
                    "rang_revenu", "rang_concurrence", "score"]].round(1))

haut = classement.head(3)
bas = classement.tail(1)
milliers = lambda x: f"{x:,.0f}".replace(",", " ")
print("Les trois arrondissements les mieux placés : "
      + " ; ".join(f"{n} (revenu médian {milliers(r)} € par an, {d:.1f} annonces pour 1 000 habitants)"
                   for n, r, d in zip(haut.index, haut["revenu_median"], haut["annonces_pour_1000_hab"])) + ".")
print(f"Le moins bien placé : {bas.index[0]} (revenu médian {milliers(bas['revenu_median'].iloc[0])} € par an).")

In [ ]:
fig_revenu = px.bar(par_arrondissement.reset_index(), x="neighbourhood_cleansed", y="revenu_median",
                    labels={"neighbourhood_cleansed": "Arrondissement", "revenu_median": "Revenu médian estimé (€ par an)"},
                    title="Le revenu médian estimé varie fortement d'un arrondissement à l'autre")
fig_revenu.show()

fig_densite = px.scatter(par_arrondissement.reset_index(), x="annonces_pour_1000_hab", y="revenu_median",
                         size="annonces_actives", text="neighbourhood_cleansed",
                         labels={"annonces_pour_1000_hab": "Annonces actives pour 1 000 habitants",
                                 "revenu_median": "Revenu médian estimé (€ par an)"},
                         title="Revenu médian et densité de concurrence par arrondissement")
fig_densite.update_traces(textposition="top center")
fig_densite.show()

### Lecture pour l'investisseur et limites
*Lecture (à valider par l'équipe après relecture des chiffres ci-dessus) :* le classement est serré (plusieurs arrondissements à égalité de score) parce que les deux critères s'opposent. Les 1er et 2e arrondissements ont les revenus médians les plus élevés mais une densité d'annonces environ trois fois supérieure à celle des autres ; les 8e et 9e ont la concurrence la plus faible mais aussi les revenus les plus bas ; les 6e, 3e et 7e offrent le meilleur compromis.
*Limites :*
- le revenu et l'occupation sont des **estimations d'Inside Airbnb** fabriquées à partir des avis (voir B1), pas des mesures ;
- la densité rapporte les annonces à la **population résidente** : elle gonfle les arrondissements centraux peu peuplés (1er, 2e) et ignore la demande touristique ;
- les prix manquants (annonces souvent non réservables) sont exclus du revenu et du prix médians ;
- ce classement ne dit rien du **coût d'achat** ni des règles de location par arrondissement.

## B3 — Q4 : licences et cadre légal

Licences et cohérence avec les règles locales de Lyon


Objectif : décrire le champ `license`, repérer les informations absentes, les formats atypiques,
les numéros réutilisés et les incohérences à examiner. **Ce travail ne calcule pas un taux d’illégalité.**
Un numéro apparemment cohérent n’est ni authentifié ni une preuve d’autorisation de changement d’usage.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import plotly.express as px

DOSSIER_B2 = SORTIES_B / 'sorties_b2_jointure'
SORTIES = SORTIES_B / 'sorties_b3_q4'
STATUT = 'validee'
DATE_REFERENCE = pd.Timestamp('2026-09-19')
DATE_CONSULTATION = '2026-11-21'
PLAFOND_RP_2026 = 90
f_b2 = DOSSIER_B2 / f'listings_avec_avis_{STATUT}.csv'
if not f_b2.is_file():
    raise FileNotFoundError(f'Export B2 absent : {f_b2.resolve()}')
base = pd.read_csv(f_b2, dtype={'id': 'string', 'host_id': 'string'})
assert len(base) > 0 and base['id'].notna().all() and base['id'].is_unique
dates = pd.to_datetime(base['last_scraped'], errors='raise')
assert dates.notna().all() and dates.eq(DATE_REFERENCE).all()
base['nb_avis_365j'] = pd.to_numeric(base['nb_avis_365j'], errors='raise')
assert base['nb_avis_365j'].notna().all() and base['nb_avis_365j'].ge(0).all()
assert base['nb_avis_365j'].mod(1).eq(0).all()
def entier(x): return f'{x:,.0f}'.replace(',', ' ')
def dec(x, n=1): return f'{x:.{n}f}'.replace('.', ',')
print('Annonces B2 :', entier(len(base)))

### 2. Récupérer les licences 
On relit seulement `id`, `license`, `property_type` et `estimated_occupancy_l365d`.
Les identifiants sont des chaînes. Les valeurs sources sont préservées.
Si plusieurs lignes d’un identifiant retenu donnent des valeurs contradictoires dans ces colonnes

In [ ]:
colonnes = ['id', 'license', 'property_type', 'estimated_occupancy_l365d']
# Champ licence depuis la table nettoyée ; « Non renseigné » (A4) correspond à un champ vide
source = listings_propre[colonnes].copy()
source['id'] = source['id'].astype('string')
source['license'] = source['license'].replace('Non renseigné', '').fillna('').astype('string')
source['property_type'] = source['property_type'].fillna('').astype('string')
source['estimated_occupancy_l365d'] = source['estimated_occupancy_l365d'].astype('string').fillna('')
source = source.loc[source['id'].isin(base['id'])].copy()
conflits = source.groupby('id')[colonnes[1:]].nunique(dropna=False).gt(1).any(axis=1)
assert not conflits.any(), 'Conflits source sur les annonces retenues : valider avec le membre 1.'
source = source.drop_duplicates('id')
df = base.merge(source, on='id', how='left', validate='one_to_one', indicator='jointure_source')
assert len(df) == len(base) and df['id'].is_unique
assert df['jointure_source'].eq('both').all(), 'Certaines annonces B2 sont absentes de la source.'
df['active_365j'] = df['nb_avis_365j'].gt(0)
print('Jointure contrôlée :', entier(len(df)), 'annonces, toutes retrouvées.')

### 3. Normaliser 
On conserve `license`, puis on retire les espaces dans une copie et on passe les lettres en majuscules.
On ne remplace ni chiffres, ni préfixes, ni tirets. Les deux derniers caractères peuvent être des lettres :
un filtre imposant 13 chiffres serait trop restrictif.

Statuts exclusifs : absent, bail mobilité déclaré, exemption hôtelière déclarée, structure compatible et
préfixe lyonnais, structure compatible mais autre préfixe, autre format à vérifier.
Les préfixes `69381` à `69389` correspondent aux arrondissements ; `69123` à la commune.
Leur présence ne valide pas la licence. Les mentions de régime ne sont pas des exemptions vérifiées.

In [ ]:
def normaliser_licence(valeur):
    texte = '' if pd.isna(valeur) else str(valeur).strip()
    return re.sub(r'\s+', '', texte).upper()

PREFIXES_LYON = {'69123'} | {f'6938{i}' for i in range(1, 10)}
vides = {'', 'NAN', 'NA', 'N/A', 'NULL', 'NONE', '<NA>'}
def classer_licence(valeur):
    texte = '' if pd.isna(valeur) else str(valeur).strip()
    cle = normaliser_licence(texte)
    if cle in vides: return 'Absent'
    if texte.casefold() == 'available with a mobility lease only ("bail mobilité")':
        return 'Bail mobilité déclaré'
    if texte.casefold() == 'exempt - hotel-type listing':
        return 'Exemption hôtelière déclarée'
    if re.fullmatch(r'[0-9]{11}[A-Z0-9]{2}', cle):
        return 'Structure et préfixe lyonnais compatibles' if cle[:5] in PREFIXES_LYON else 'Structure compatible, autre préfixe'
    return 'Autre format à vérifier'

ordre = ['Structure et préfixe lyonnais compatibles', 'Structure compatible, autre préfixe',
         'Autre format à vérifier', 'Absent', 'Bail mobilité déclaré', 'Exemption hôtelière déclarée']
df['licence_normalisee'] = df['license'].map(normaliser_licence).astype('string')
df['statut_licence'] = df['license'].map(classer_licence)
df['structure_compatible'] = df['licence_normalisee'].str.fullmatch(r'[0-9]{11}[A-Z0-9]{2}').fillna(False)
df['prefixe_licence'] = df['licence_normalisee'].str[:5].where(df['structure_compatible'])
assert df['statut_licence'].isin(ordre).all()
# Contrôles utiles : absence, clé alphanumérique et préfixe étranger au référentiel lyonnais.
assert classer_licence('') == 'Absent'
assert classer_licence('69381567890AB') == ordre[0]
assert classer_licence('1234567890123') == ordre[1]
assert classer_licence('0') == ordre[2]
bilan = df['statut_licence'].value_counts().reindex(ordre, fill_value=0).rename_axis('statut').reset_index(name='nb_annonces')
bilan['part_pct'] = 100 * bilan['nb_annonces'] / len(df)
assert bilan['nb_annonces'].sum() == len(df)
print(bilan.round(2).to_string(index=False))
fig_statuts = px.bar(bilan, y='statut', x='nb_annonces', orientation='h', text='nb_annonces',
    template='plotly_white', title='Licences : informations renseignées et statuts à vérifier',
    labels={'statut': '', 'nb_annonces': 'Nombre d’annonces'}, hover_data={'part_pct': ':.1f'})
fig_statuts.update_yaxes(categoryorder='array', categoryarray=ordre[::-1])
fig_statuts.show()

### 4. Comparer les périmètres avec des dénominateurs explicites
On harmonise les variantes de `room_type` et des noms d’arrondissement dans des colonnes séparées.
Les logements entiers actifs constituent un périmètre utile pour l’examen documentaire, sans présumer
qu’ils sont tous des résidences principales ou tous soumis au même régime.
Le tableau croise les statuts pour l’ensemble, les annonces actives et les logements entiers actifs.
Les pourcentages sont calculés dans chacun de ces périmètres, jamais en supprimant les licences absentes.

In [ ]:
types = {'entire home/apt':'Logement entier', 'entire home':'Logement entier',
         'private room':'Chambre privée', 'private':'Chambre privée',
         'shared room':'Chambre partagée', 'hotel room':'Chambre d’hôtel'}
df['type_harmonise'] = df['room_type'].astype('string').str.strip().str.lower().map(types).fillna('Type non reconnu')
df['arrondissement'] = df['neighbourhood_cleansed'].astype('string').str.strip().str.lower().str.extract(
    r'^([1-9])(?:er|e)\s+arrondissement$', expand=False).astype('Int64')
df['quartier_affiche'] = df['arrondissement'].astype('string').fillna('Non reconnu')
perimetres = {'Toutes les annonces B2': pd.Series(True, index=df.index),
              'Actives selon les avis': df['active_365j'],
              'Logements entiers actifs': df['active_365j'] & df['type_harmonise'].eq('Logement entier')}
tableaux=[]
for nom, masque in perimetres.items():
    t=df.loc[masque, 'statut_licence'].value_counts().reindex(ordre, fill_value=0).rename_axis('statut').reset_index(name='nb_annonces')
    t['perimetre']=nom; t['denominateur']=int(masque.sum())
    t['part_pct']=100*t['nb_annonces']/int(masque.sum()) if masque.any() else np.nan
    assert t['nb_annonces'].sum()==masque.sum()
    tableaux.append(t)
comparaison=pd.concat(tableaux,ignore_index=True)
print(comparaison.round(2).to_string(index=False))
print('Types non reconnus :', df['type_harmonise'].eq('Type non reconnu').sum(),
      '; arrondissements non reconnus :', df['arrondissement'].isna().sum())
fig_perimetres=px.bar(comparaison, x='perimetre', y='part_pct', color='statut',
    category_orders={'statut':ordre}, template='plotly_white',
    title='Répartition du champ licence selon le périmètre',
    labels={'perimetre':'', 'part_pct':'Part des annonces (%)','statut':'Statut descriptif'},
    hover_data=['nb_annonces','denominateur'])
fig_perimetres.update_yaxes(range=[0,100]);fig_perimetres.show()

### 4 bis. Lien avec la durée minimale affichée et le type de logement

On croise le champ licence avec `minimum_nights` et `room_type`.
Les classes **moins de 30 nuits**, **30 à 364 nuits** et **365 nuits ou plus** sont des catégories descriptives,
pas des catégories juridiques. Le minimum affiché n’est ni la durée du séjour réel ni celle du contrat.
Un mois calendaire ne fait pas toujours 30 jours ; le seuil ne permet donc pas de conclure à un bail mobilité
ou à une location touristique. Les durées non exploitables restent visibles dans les tableaux et le graphique.

Un minimum élevé ou une chambre peut correspondre à une situation différente d’un logement entier touristique,
mais ne prouve pas une exemption. Ces croisements montrent l’hétérogénéité des annonces sans licence renseignée,
pas leur conformité. Les logements entiers actifs, au minimum affiché inférieur à 30 nuits et sans licence,
forment un **segment à examiner**, sans estimation de probabilité d’infraction.


In [ ]:
# AJOUT — Lien avec la durée minimale et le type de logement (2e point de la consigne de Q4)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

SEUIL_COURTE, SEUIL_LONGUE = 30, 365
minimum_affiche = pd.to_numeric(df['minimum_nights'], errors='coerce')
# Même prudence que B1 : une valeur manquante, non entière ou négative n'est pas classée.
minimum_classable = minimum_affiche.where(minimum_affiche.ge(1) & np.isfinite(minimum_affiche) & minimum_affiche.mod(1).eq(0))
libelles_duree = ['Minimum < 30 nuits', 'Minimum de 30 à 364 nuits', 'Minimum de 365 nuits ou plus']
df['categorie_duree'] = pd.cut(minimum_classable, [0, SEUIL_COURTE, SEUIL_LONGUE, np.inf],
                               right=False, labels=libelles_duree)

df['duree_affichee_groupe'] = df['categorie_duree'].cat.add_categories(['Non classable']).fillna('Non classable')

tab_duree = pd.crosstab(df['statut_licence'], df['duree_affichee_groupe'], margins=True,
                        margins_name='Total').reindex(ordre + ['Total'], fill_value=0)
tab_type = pd.crosstab(df['statut_licence'], df['type_harmonise'], margins=True,
                       margins_name='Total').reindex(ordre + ['Total'], fill_value=0)
print('Statut de la licence × durée minimale affichée :')
print(tab_duree.to_string())
print('\nStatut de la licence × type de logement :')
print(tab_type.to_string())
print('Durée minimale non classable :', int(df['categorie_duree'].isna().sum()))
print('Les pourcentages portant sur les durées classées excluent ces valeurs non classables.')

assert tab_duree.loc['Total', 'Total'] == len(df)
assert tab_type.loc['Total', 'Total'] == len(df)
# Croisements descriptifs : les régimes juridiques ne sont pas identifiés.
def ratio_pct(numerateur, denominateur):
    return 100 * numerateur / denominateur if denominateur else np.nan
absent = df['statut_licence'].eq('Absent')
courte = df['categorie_duree'].eq(libelles_duree[0])
part_absents_30_plus = df.loc[absent & df['categorie_duree'].notna(), 'categorie_duree'].ne(libelles_duree[0]).mean()
part_longue_absent = df.loc[df['categorie_duree'].eq(libelles_duree[2]), 'statut_licence'].eq('Absent').mean()
part_chambres = df.loc[absent & courte, 'type_harmonise'].isin(['Chambre privée', 'Chambre partagée']).mean()
segment = courte & df['type_harmonise'].eq('Logement entier') & df['active_365j']
n_entiers_courts_absents = int((segment & absent).sum())
part_courte_lyon = df.loc[courte, 'statut_licence'].eq(ordre[0]).mean()
print(f"\nAnnonces au champ absent qui imposent au moins 30 nuits : {dec(100 * part_absents_30_plus)} %")
print(f"Annonces au minimum affiché >= 365 nuits avec champ absent : {dec(100 * part_longue_absent)} %")
print(f"Annonces de courte durée au champ absent qui sont des chambres : {dec(100 * part_chambres)} %")
print(f"Logements entiers, courte durée, actifs, champ absent : {entier(n_entiers_courts_absents)} "
      f"sur {entier(segment.sum())} ({dec(ratio_pct(n_entiers_courts_absents, int(segment.sum())))} %)")

# Graphique Matplotlib (compte pour B4) : titre = constat
repartition = pd.crosstab(df['duree_affichee_groupe'], df['statut_licence'], normalize='index').reindex(columns=ordre, fill_value=0)
couleurs = ['#087F8C', '#7FB8BF', '#C9DEE0', '#C0392B', '#E1A95F', '#C9A0DC']
fig_duree, ax = plt.subplots(figsize=(11, 4.8))
gauche = np.zeros(len(repartition))
etiquettes = [c.replace(' (', '\n(') for c in repartition.index.astype(str)]
for statut, couleur in zip(ordre, couleurs):
    valeurs = repartition[statut].to_numpy()
    ax.barh(etiquettes, valeurs, left=gauche, color=couleur, label=statut)
    for y, (v, g) in enumerate(zip(valeurs, gauche)):
        if v >= 0.06:
            ax.text(g + v / 2, y, f'{100 * v:.0f} %', ha='center', va='center', fontsize=8,
                    color='white' if couleur in ('#087F8C', '#C0392B') else '#222')
    gauche += valeurs
ax.set_title(f"{100 * part_courte_lyon:.0f} % des minima < 30 nuits : structure et préfixe compatibles ;\n"
             f"champ absent pour {100 * part_longue_absent:.0f} % des minima de 365 nuits ou plus",
             loc='left', fontsize=11)
ax.set_xlabel('Part des annonces de la catégorie (%)')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlim(0, 1)
ax.invert_yaxis()
ax.spines[['top', 'right']].set_visible(False)
fig_duree.legend(loc='lower center', ncol=3, frameon=False, fontsize=8)
plt.tight_layout(rect=[0, 0.14, 1, 1])
plt.show()

### 5. Repérer les numéros réutilisés et les écarts géographiques
On compte les identifiants d’annonce distincts par numéro de structure compatible. Les mentions d’hôtel
ou de bail mobilité ne sont pas traitées comme des numéros identiques partagés.

Une répétition peut correspondre à plusieurs annonces d’un même logement, à une gestion commune,
à une erreur de saisie ou à une altération pédagogique. **Ce n’est pas une preuve de fraude.**
On confronte aussi le préfixe d’arrondissement au quartier déclaré. Le code communal `69123`
ne permet pas de vérifier l’arrondissement. Les adresses légales et le registre ne sont pas disponibles.

In [ ]:
numeros=df.loc[df['structure_compatible']].copy()
reutilisation=numeros.groupby('licence_normalisee').agg(nb_annonces=('id','nunique'),
    nb_hotes=('host_id','nunique'), nb_arrondissements=('arrondissement','nunique')).reset_index()
reutilisation=reutilisation.loc[reutilisation['nb_annonces'].gt(1)].sort_values('nb_annonces',ascending=False)
df['numero_reutilise']=df['structure_compatible'] & df['licence_normalisee'].isin(reutilisation['licence_normalisee'])
df['arrondissement_licence']=pd.to_numeric(df['prefixe_licence'].str.extract(r'^6938([1-9])$',expand=False),errors='coerce').astype('Int64')
df['geographie_comparable']=df['arrondissement_licence'].notna() & df['arrondissement'].notna()
df['ecart_arrondissement']=(df['geographie_comparable'] & df['arrondissement_licence'].ne(df['arrondissement'])).fillna(False)
n_geo=int(df['geographie_comparable'].sum())
print('Numéros réutilisés :',len(reutilisation),'; annonces concernées :',int(df['numero_reutilise'].sum()))
print('Numéros réutilisés entre plusieurs hôtes :',int(reutilisation['nb_hotes'].gt(1).sum()))
print('Écarts d’arrondissement :',int(df['ecart_arrondissement'].sum()),'sur',n_geo,'annonces comparables.')
assert not df.loc[~df['structure_compatible'],'numero_reutilise'].any()

### 6. Examiner les régimes déclarés et la durée estimée
Pour le **cas général** du bail mobilité, une durée minimale affichée inférieure à 28 nuits est un
signal à examiner : un mois calendaire n’a pas toujours 30 jours. Le paramètre `minimum_nights`
n’est pas le contrat signé ; il ne prouve ni le régime applicable ni une violation de sa durée.
Le statut hôtelier ne peut pas être confirmé par `room_type` ou `property_type` seuls.

`estimated_occupancy_l365d` est une estimation de nuits sur 365 jours glissants. On peut décrire son
dépassement numérique de 90, mais **pas conclure au dépassement du plafond légal** : celui-ci concerne
les résidences principales sur une année civile. La fenêtre chevauche 2025 et 2026, le statut du logement
et les dérogations sont inconnus. `availability_365` n’est pas un nombre de nuits effectivement louées.

In [ ]:
minimum=pd.to_numeric(df['minimum_nights'],errors='coerce')
minimum_valide=minimum.notna() & np.isfinite(minimum) & minimum.ge(1) & minimum.mod(1).eq(0)
df['mobilite_min_court']=df['statut_licence'].eq('Bail mobilité déclaré') & minimum_valide & minimum.lt(28)
df['mobilite_min_inexploitable']=df['statut_licence'].eq('Bail mobilité déclaré') & ~minimum_valide
occupation=pd.to_numeric(df['estimated_occupancy_l365d'],errors='coerce')
occupation_valide=occupation.notna() & np.isfinite(occupation) & occupation.between(0,365)
df['occupation_estimee_exploitable']=occupation_valide
df['estimation_plus_90']=occupation.gt(PLAFOND_RP_2026).where(occupation_valide).astype('boolean')
entiers=df['type_harmonise'].eq('Logement entier')
n_occ=int((entiers & occupation_valide).sum())
n_plus90=int((entiers & df['estimation_plus_90'].fillna(False)).sum())
print('Bail mobilité déclaré avec minimum affiché < 28 nuits :',int(df['mobilite_min_court'].sum()))
print('Bail mobilité déclaré avec minimum inexploitable :',int(df['mobilite_min_inexploitable'].sum()))
print('Logements entiers avec estimation > 90 nuits sur 365 jours :',n_plus90,'sur',n_occ,'estimations exploitables.')
print('Ce dernier indicateur ne mesure PAS une infraction au plafond annuel des résidences principales.')

### 7. Construire une liste de vérifications, sans score d’illégalité
Les indicateurs ci-dessous sont distincts et peuvent se cumuler ; leurs nombres ne doivent pas être additionnés
pour obtenir un nombre d’annonces. Le total unique est calculé avec une union des indicateurs.
Le dépassement de 90 nuits **estimées** n’entre pas dans cette liste documentaire : les données ne permettent
pas d’appliquer le plafond légal. La présence d’un hôtel ou d’un bail mobilité n’est pas en soi une anomalie.

In [ ]:
df['licence_absente']=df['statut_licence'].eq('Absent')
df['format_a_verifier']=df['statut_licence'].eq('Autre format à vérifier')
df['prefixe_a_verifier']=df['statut_licence'].eq('Structure compatible, autre préfixe')
flags={'licence_absente':'Champ licence absent', 'format_a_verifier':'Autre format à vérifier',
       'prefixe_a_verifier':'Préfixe hors référentiel lyonnais', 'numero_reutilise':'Numéro réutilisé',
       'ecart_arrondissement':'Écart d’arrondissement', 'mobilite_min_court':'Bail mobilité / minimum < 28 nuits',
       'mobilite_min_inexploitable':'Bail mobilité / minimum inexploitable'}
df['verification_documentaire']=df[list(flags)].any(axis=1)
df['nb_signaux']=df[list(flags)].sum(axis=1)
audit_signaux=pd.DataFrame([{'signal':libelle,'nb_annonces':int(df[col].sum())} for col,libelle in flags.items()])
audit_signaux['part_toutes_annonces_pct']=100*audit_signaux['nb_annonces']/len(df)
par_quartier=df.groupby('quartier_affiche').agg(nb_annonces=('id','size'),
    licences_absentes=('licence_absente','sum')).reset_index()
par_quartier['part_absentes_pct']=100*par_quartier['licences_absentes']/par_quartier['nb_annonces']
assert par_quartier['nb_annonces'].sum()==len(df)
print(audit_signaux.round(2).to_string(index=False))
print('Annonces distinctes avec au moins un signal documentaire :',int(df['verification_documentaire'].sum()))
fig_signaux=px.bar(audit_signaux, x='nb_annonces', y='signal',orientation='h',text='nb_annonces',
    title='Signaux documentaires non exclusifs — aucune qualification d’illégalité',template='plotly_white',
    labels={'signal':'','nb_annonces':'Nombre d’annonces'})
fig_signaux.show()
fig_quartiers=px.bar(par_quartier, x='quartier_affiche', y='part_absentes_pct',text_auto='.1f',
    template='plotly_white',title='Part des annonces dont le champ licence est absent, par arrondissement',
    labels={'quartier_affiche':'Arrondissement','part_absentes_pct':'Champ licence absent (%)'},
    hover_data=['nb_annonces','licences_absentes'])
fig_quartiers.update_yaxes(range=[0,100]);fig_quartiers.show()

### 8. Conclusion  et limites
Le bilan décrit ce que contient le fichier, pas le registre municipal. Les annonces sans signal peuvent aussi présenter des irrégularités non observables. Les différences entre quartiers peuvent refléter la composition de l’offre
(hôtels, chambres, locations longues) et la qualité de saisie, pas seulement les pratiques des hôtes.

In [ ]:
effectifs=bilan.set_index('statut')['nb_annonces']
n_abs=int(effectifs['Absent']);n_comp=int(effectifs[ordre[0]])
n_atyp=int(effectifs[ordre[1]]+effectifs[ordre[2]])
conclusion=(f'Sur {entier(len(df))} annonces retenues en B2, {entier(n_abs)} '
    f'({dec(100*n_abs/len(df))} %) ont un champ licence absent. '
    f'{entier(n_comp)} présentent une structure et un préfixe lyonnais compatibles avec le contrôle descriptif. '
    'Cela ne prouve ni leur authenticité ni une autorisation de changement d’usage.\n\n'
    f'{entier(effectifs["Bail mobilité déclaré"])} annonces déclarent un bail mobilité et '
    f'{entier(effectifs["Exemption hôtelière déclarée"])} une exemption hôtelière. Ces mentions restent à justifier. '
    f'{entier(n_atyp)} valeurs présentent un autre format ou un préfixe hors du référentiel lyonnais.\n\n'
    f'{entier(len(reutilisation))} numéros de structure compatible sont partagés par '
    f'{entier(df["numero_reutilise"].sum())} annonces distinctes. '
    f'{entier(df["ecart_arrondissement"].sum())} écarts d’arrondissement apparaissent parmi '
    f'{entier(n_geo)} annonces comparables. Ces signaux appellent une vérification, sans démontrer une fraude.\n\n'
    f'{entier(df["mobilite_min_court"].sum())} annonces déclarant un bail mobilité affichent un minimum inférieur '
    'à 28 nuits : examiner les contrats et le régime applicable avant de conclure. '
    f'{entier(n_plus90)} logements entiers sur {entier(n_occ)} estimations exploitables dépassent '
    'numériquement 90 nuits estimées sur 365 jours. Ce chiffre ne mesure pas le non-respect du plafond de Lyon : '
    'les résidences principales et les jours réellement loués par année civile ne sont pas identifiés.\n\n'
    'Aucun taux de conformité ou d’illégalité ne peut être déduit de ce fichier seul. '
    'La vérification documentaire doit compléter les contrôles de données. '
    'Résultats pédagogiques provisoires, à recalculer après validation du nettoyage.')
print(conclusion)

### 8 bis. Conséquence pour l’investisseur

Le bail mobilité n’est pas une solution automatiquement accessible à tout locataire. Aucun classement
de rendement entre location touristique, mobilité et longue durée n’est démontré ici : il faut comparer
les loyers ou prix, les périodes sans location et les coûts sur des hypothèses cohérentes.

**Sanctions, pour la section « Risques » de E2.** Changement d'usage illicite : amende civile jusqu'à 100 000 € par logement. Dépassement du plafond de la résidence principale : jusqu'à 15 000 € (ministère de la Transition écologique, FAQ sur la loi n° 2024-1039, https://www.ecologie.gouv.fr/sites/default/files/documents/FAQ%20sur%20l%27application%20dans%20le%20temps%20de%20la%20preuve%20de%20l%27usage%20%C3%A0%20la%20suite%20de%20la%20loi%20n%C2%B0%202024-1039%20du%2019%20novembre%202024_1.pdf).

Ces montants sont des **plafonds de sanctions civiles**, et non des montants automatiques par annonce signalée.
La FAQ officielle de juin 2025 distingue bien le plafond de location et le changement d’usage (S7).
Aucune amende théorique n’est calculée à partir de nos anomalies de données.


In [ ]:
# Scénarios à vérifier : aucune attribution automatique du régime juridique.
modeles = pd.DataFrame([
    {'modele': 'Location touristique de sa résidence principale',
     'conditions_lyon': 'Enregistrement et plafond de 90 jours par année civile ; vérifier le statut réel et les exceptions (S1, S6)',
     'indicateur_disponible': 'Aucun effectif fiable de résidences principales',
     'lecture': 'Non assimilable à un bien acheté exclusivement pour la location sans y habiter.'},
    {'modele': 'Location touristique d’un autre logement',
     'conditions_lyon': 'Changement d’usage préalable selon la situation ; conditions locales de compensation et enregistrement (S1, S3)',
     'indicateur_disponible': 'Aucun effectif fiable de biens d’investissement ou de logements autorisés',
     'lecture': 'Plus de 90 nuits estimées ne suffit pas à identifier ce modèle ni sa conformité.'},
    {'modele': 'Bail mobilité',
     'conditions_lyon': 'Vérifier le contrat, le locataire éligible et les conditions applicables ; cas général de 1 à 10 mois (S4)',
     'indicateur_disponible': f'{int(df["statut_licence"].eq("Bail mobilité déclaré").sum())} mentions déclarées dans license',
     'lecture': 'Déclaration non vérifiée ; la classe 30–364 nuits ne permet pas d’identifier un bail mobilité.'},
    {'modele': 'Bail d’habitation meublé classique',
     'conditions_lyon': 'Vérifier les règles du bail, du logement et du loyer applicables au cas précis',
     'indicateur_disponible': f'{int(df["categorie_duree"].eq(libelles_duree[2]).sum())} minima affichés de 365 nuits ou plus, sans preuve de contrat',
     'lecture': 'Rendement et risques à évaluer ; aucune supériorité ou infériorité établie par cette analyse.'}
])
print(modeles.to_string(index=False))
conclusion += (
    '\n\nCroisement avec la durée minimale affichée : parmi les annonces au champ licence absent '
    f'et à durée classable, {dec(100 * part_absents_30_plus)} % affichent un minimum d’au moins 30 nuits. '
    f'Parmi celles dont le minimum atteint 365 nuits, {dec(100 * part_longue_absent)} % ont un champ absent. '
    f'Parmi les minima inférieurs à 30 nuits sans licence renseignée, {dec(100 * part_chambres)} % sont des chambres. '
    'Ces croisements montrent qu\u2019un champ licence vide ne signifie pas forcément une location illégale : '
    'une grande partie des annonces sans licence affichent un minimum d\u2019au moins 30 nuits ou proposent une chambre, '
    'des situations différentes du logement entier loué à la nuitée. '
    'Ces paramètres ne déterminent ni le contrat réel ni une exemption réglementaire. '
    f'{entier(df["categorie_duree"].isna().sum())} durées non classables restent visibles séparément. '
    f'Le segment des logements entiers actifs au minimum inférieur à 30 nuits contient '
    f'{entier(n_entiers_courts_absents)} champs licence absents sur {entier(segment.sum())} annonces '
    f'({dec(ratio_pct(n_entiers_courts_absents, int(segment.sum())))} %). '
    'Ce segment mérite une vérification documentaire, sans probabilité d’illégalité calculable.'
    '\n\nPour l\u2019investisseur, la réponse est conditionnelle. Louer à la nuitée un bien acheté pour la location, '
    'qui n\u2019est donc pas sa résidence principale, n\u2019est possible à Lyon qu\u2019avec une autorisation préalable '
    'de changement d\u2019usage, que le numéro d\u2019enregistrement ne remplace pas, souvent avec compensation (S1, S3). '
    'C\u2019est la première condition à vérifier, avant tout calcul de rentabilité. Sans cette autorisation, restent '
    'les baux d\u2019habitation : le bail mobilité si le locataire est éligible (S4), ou le bail meublé classique. '
    'Le fichier ne permet pas de dire lequel rapporte le plus : C3 et E2 doivent chiffrer au moins un scénario '
    '« nuitée avec autorisation » et un scénario « bail d\u2019habitation ».'
)
print('\n'+conclusion)


### 9. Sources et exports

Sources officielles consultées le 21 novembre 2026 :

- **S1** — [Ville de Lyon — Déclarer un meublé de tourisme](https://www.lyon.fr/demarche/logement-habitat/declarer-un-meuble-de-tourisme)
- **S2** — [Légifrance — D324-1-1, version affichée au 27 avril 2026](https://www.legifrance.gouv.fr/codes/article_lc/LEGIARTI000039475092/2026-04-27)
- **S3** — [Ville de Lyon — Changement d’usage](https://www.lyon.fr/demarche/logement-habitat/demander-le-changement-dusage-dun-logement-0)
- **S4** — [Service Public — Bail mobilité](https://www.service-public.gouv.fr/particuliers/vosdroits/F34759)
- **S5** — [Insee — COG 2026, Lyon et ses neuf arrondissements](https://www.insee.fr/fr/metadonnees/geographie/commune/69123-lyon)
- **S6** — [Conseil du 5e arrondissement, 12 juin 2025 — plafond applicable au 1er janvier 2026](https://lyon-webdelib.digitechcloud.fr/webdelibplus_Central/jsp/showFile.jsp?datePub=30%2F06%2F2025&dateRetLega=27%2F06%2F2025&pdf=IUvl3cQtSbjRCJgky5xw5vPKcB3KcHGx2NHT34Abd4rEudxktLCuH6lDc6T4iJhXDTZMfk6kp%2BjXAqBLwH6I6SYox5X%2Fj0DZx0tp%2Bsem530Cv%2B4KwaPKA7%2BTbFpMlatfjyQBpSdqUSv5OyRNa2Q2qMRpRvW5EmVjf%2F1zyBEct5aas64wY6zTSA%3D%3D)


- **S7** — [Ministère de la Transition écologique, FAQ de juin 2025 : sanctions distinctes](https://www.ecologie.gouv.fr/sites/default/files/documents/FAQ%20sur%20l%27application%20dans%20le%20temps%20de%20la%20preuve%20de%20l%27usage%20%C3%A0%20la%20suite%20de%20la%20loi%20n%C2%B0%202024-1039%20du%2019%20novembre%202024_1.pdf)


In [ ]:
import html
import io
import base64
SORTIES.mkdir(parents=True,exist_ok=True)
sources = [('S1', 'Ville de Lyon — Déclarer un meublé de tourisme', 'https://www.lyon.fr/demarche/logement-habitat/declarer-un-meuble-de-tourisme'), ('S2', 'Légifrance — D324-1-1, version affichée au 27 avril 2026', 'https://www.legifrance.gouv.fr/codes/article_lc/LEGIARTI000039475092/2026-04-27'), ('S3', 'Ville de Lyon — Changement d’usage', 'https://www.lyon.fr/demarche/logement-habitat/demander-le-changement-dusage-dun-logement-0'), ('S4', 'Service Public — Bail mobilité', 'https://www.service-public.gouv.fr/particuliers/vosdroits/F34759'), ('S5', 'Insee — COG 2026, Lyon et ses neuf arrondissements', 'https://www.insee.fr/fr/metadonnees/geographie/commune/69123-lyon'), ('S6', 'Conseil du 5e arrondissement, 12 juin 2025 — plafond applicable au 1er janvier 2026', 'https://lyon-webdelib.digitechcloud.fr/webdelibplus_Central/jsp/showFile.jsp?datePub=30%2F06%2F2025&dateRetLega=27%2F06%2F2025&pdf=IUvl3cQtSbjRCJgky5xw5vPKcB3KcHGx2NHT34Abd4rEudxktLCuH6lDc6T4iJhXDTZMfk6kp%2BjXAqBLwH6I6SYox5X%2Fj0DZx0tp%2Bsem530Cv%2B4KwaPKA7%2BTbFpMlatfjyQBpSdqUSv5OyRNa2Q2qMRpRvW5EmVjf%2F1zyBEct5aas64wY6zTSA%3D%3D')]
sources.append(('S7', 'Ministère de la Transition écologique — FAQ de juin 2025', 'https://www.ecologie.gouv.fr/sites/default/files/documents/FAQ%20sur%20l%27application%20dans%20le%20temps%20de%20la%20preuve%20de%20l%27usage%20%C3%A0%20la%20suite%20de%20la%20loi%20n%C2%B0%202024-1039%20du%2019%20novembre%202024_1.pdf'))
sources_df=pd.DataFrame(sources,columns=['reference','titre','url'])
sources_df['date_consultation']=DATE_CONSULTATION
audit_cols=['id','host_id','license','licence_normalisee','statut_licence','type_harmonise',
    'property_type','arrondissement','active_365j','geographie_comparable','arrondissement_licence',
    'duree_affichee_groupe','occupation_estimee_exploitable','estimation_plus_90','verification_documentaire','nb_signaux']+list(flags)
tables={'audit_licences_interne':df[audit_cols], 'bilan_licences':bilan,
    'comparaison_perimetres':comparaison,'numeros_reutilises_interne':reutilisation,
    'signaux_documentaires':audit_signaux,'licences_par_arrondissement':par_quartier,'sources_reglementaires':sources_df}
tables.update({'croisement_licence_duree':tab_duree.reset_index(),
               'croisement_licence_type':tab_type.reset_index(), 'scenarios_investisseur':modeles})
fig_duree.savefig(SORTIES/'q4_licences_duree.png',dpi=150,bbox_inches='tight')
for nom,t in tables.items():t.to_csv(SORTIES/f'{nom}_{STATUT}.csv',index=False,encoding='utf-8-sig')
(SORTIES/f'conclusion_q4_{STATUT}.txt').write_text(conclusion,encoding='utf-8')
figures=[fig_statuts,fig_perimetres,fig_signaux,fig_quartiers]
blocs=[f.to_html(full_html=False,include_plotlyjs=(i==0)) for i,f in enumerate(figures)]
tampon=io.BytesIO()
fig_duree.savefig(tampon,format='png',dpi=150,bbox_inches='tight')
image_duree=base64.b64encode(tampon.getvalue()).decode('ascii')
blocs.append('<h2>Durée minimale affichée et licence</h2><img style="max-width:100%;height:auto" '
             f'alt="Répartition des licences selon le minimum affiché" src="data:image/png;base64,{image_duree}">')
blocs.append('<h2>Scénarios à vérifier</h2>'+modeles.to_html(index=False,escape=True))
liens=''.join(f'<li>{html.escape(k)} — <a href="{html.escape(u,quote=True)}">{html.escape(t)}</a></li>' for k,t,u in sources)
page=('<!doctype html><html lang="fr"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">'
    '<title>Lyon — Licences et règles locales</title><style>body{font-family:Arial,sans-serif;max-width:1150px;margin:30px auto;padding:0 20px;color:#243444}'
    'p,li{line-height:1.6}.note{padding:18px;background:#eef5f6}</style></head><body>'
    '<h1>B3 — Q4 : licences et règles locales à Lyon</h1><p class="note">Résultats provisoires : '
    'contrôles de données et vérifications documentaires, sans qualification d’illégalité.</p>'
    + ''.join(blocs)+'<h2>Conclusion et limites</h2><p>'+html.escape(conclusion).replace('\n\n','</p><p>')
    +'</p><h2>Sources officielles</h2><p>Consultées le 21/11/2026 ; données collectées le 19/09/2026. '
    'S2 : version du 27/04/2026, utilisée comme repère de structure. S6 : acte du conseil du 5e arrondissement.</p><ul>'+liens
    +'</ul><p>Données Inside Airbnb, CC BY 4.0, altérées pour le cours.</p></body></html>')
(SORTIES/'B3_Q4_graphiques_Lyon.html').write_text(page,encoding='utf-8')
print('Exports :',SORTIES.resolve())

## B4 — Visualisations exploratoires (à compléter)

À produire : au moins 6 graphiques Matplotlib ou Seaborn de types différents (distribution, boxplot, barplot, heatmap de corrélation, série temporelle, nuage de points, grille). Chaque graphique a un titre qui énonce le constat, des axes nommés avec unités, et deux lignes de commentaire.
Déjà présents dans ce notebook en Matplotlib : le profil saisonnier (Q1) et la sensibilité à la définition d'activité (Q1).